# EVTEAS-Py 2.0 (Rev. 187)

Framework computacional para Estudo de Viabilidade Técnica, Econômica, Ambiental e Social (EVTEAS) aplicado à piscicultura — dissertação de Rafael Alves Bastos (UFF/MESC).

Este notebook é autocontido: reúne o código do pacote `evteas_py` em células, na ordem Input → Processamento → Output → Verificação. As premissas do caso-base são ilustrativas e devem ser substituídas pelos dados do projeto avaliado e calibradas na validação com especialistas.

In [ ]:
# Dependências (no Colab, apenas numpy-financial costuma faltar)
%pip install -q numpy pandas matplotlib openpyxl numpy-financial

In [ ]:
from __future__ import annotations
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

## 1. Configuração, premissas e rastreabilidade das fontes

In [ ]:
"""Estruturas de configuração do EVTEAS-Py.

Todas as premissas do estudo ficam concentradas em um único objeto
``EVTEASConfig`` que atravessa o pipeline (Capítulo 4, Seção 4.1). Cada bloco
corresponde a um eixo de modelagem descrito na Seção 2.8.6 da dissertação:
infraestrutura, parâmetros biológicos e eficiência de insumos (técnico);
CAPEX/OPEX, receitas e parâmetros financeiros (econômico); gestão hídrica,
energética e conformidade (ambiental); emprego, segurança e relação com a
comunidade (social); governança (G do ESG); pesos e incerteza.

Os valores numéricos *default* são ilustrativos. A origem de cada premissa é
registrada em ``EVTEASConfig.fontes`` (rastreabilidade exigida na Seção 3.6).
"""

from dataclasses import dataclass, field, fields, is_dataclass, asdict
from typing import Any, Dict, List, Optional

import numpy as np

VERSAO = "EVTEAS-Py 2.0 (Rev. 187)"

# ---------------------------------------------------------------------------
# Rastreabilidade das premissas (Seção 3.6)
# ---------------------------------------------------------------------------

CATEGORIAS_FONTE = {
    "historico": "Dado histórico do empreendimento",
    "secundario": "Dado secundário (estatística oficial, relatório setorial)",
    "cotacao": "Cotação de mercado",
    "normativo": "Valor normativo (lei, resolução, norma técnica)",
    "bibliografico": "Parâmetro bibliográfico",
    "analogo": "Empreendimento análogo",
    "especialista": "Estimativa de especialista",
    "autor": "Estimativa do autor (a validar na V&V)",
}


@dataclass
class Fonte:
    """Origem declarada de uma premissa."""
    categoria: str = "autor"
    referencia: str = ""

    def validar(self) -> "Fonte":
        if self.categoria not in CATEGORIAS_FONTE:
            raise ValueError(f"Categoria de fonte desconhecida: {self.categoria}")
        return self


@dataclass
class Distribuicao:
    """Distribuição de incerteza de uma variável de entrada.

    ``uniforme`` quando apenas mínimo e máximo são conhecidos; ``triangular``
    quando também há um valor mais provável (Oliveira; Medeiros Neto, 2012);
    ``normal`` truncada em [mínimo, máximo] e ``fixa`` para desligar a
    incerteza de uma variável sem removê-la do registro.
    """
    tipo: str = "triangular"
    minimo: float = 0.0
    moda: float = 0.0
    maximo: float = 0.0

    def validar(self) -> "Distribuicao":
        t = self.tipo.lower()
        if t not in {"triangular", "uniforme", "normal", "fixa"}:
            raise ValueError(f"Distribuição não suportada: {self.tipo}")
        if self.minimo > self.maximo:
            raise ValueError("mínimo não pode ser maior que máximo")
        if t == "triangular" and not (self.minimo <= self.moda <= self.maximo):
            raise ValueError("na triangular, mínimo <= moda <= máximo")
        return self

    def amostrar(self, rng: np.random.Generator, n: int) -> np.ndarray:
        self.validar()
        t = self.tipo.lower()
        if t == "fixa" or self.maximo == self.minimo:
            return np.full(n, self.moda if t != "uniforme" else self.minimo, dtype=float)
        if t == "triangular":
            return rng.triangular(self.minimo, self.moda, self.maximo, n)
        if t == "uniforme":
            return rng.uniform(self.minimo, self.maximo, n)
        mu = self.moda if self.minimo <= self.moda <= self.maximo else (self.minimo + self.maximo) / 2
        sigma = max((self.maximo - self.minimo) / 3.92, 1e-12)  # ±1,96σ ~ [mín, máx]
        return np.clip(rng.normal(mu, sigma, n), self.minimo, self.maximo)

    def media(self) -> float:
        t = self.tipo.lower()
        if t == "triangular":
            return (self.minimo + self.moda + self.maximo) / 3
        if t == "uniforme":
            return (self.minimo + self.maximo) / 2
        return self.moda


# ---------------------------------------------------------------------------
# Dimensão técnica — infraestrutura, biologia e eficiência de insumos
# ---------------------------------------------------------------------------

# Referências indicativas por sistema produtivo (a calibrar pelo especialista em
# piscicultura). capacidade_suporte em kg/m³ ao final do ciclo; renovação em %
# do volume ao dia; energia em kWh por kg produzido (aeração + bombeamento).
SISTEMAS_PRODUTIVOS = {
    "extensivo": {"capacidade_suporte_kg_m3": 0.3, "renovacao_pct_dia": 1.0, "energia_kwh_kg": 0.05},
    "semi-intensivo": {"capacidade_suporte_kg_m3": 1.2, "renovacao_pct_dia": 3.0, "energia_kwh_kg": 0.60},
    "intensivo": {"capacidade_suporte_kg_m3": 3.0, "renovacao_pct_dia": 5.0, "energia_kwh_kg": 1.20},
    "superintensivo": {"capacidade_suporte_kg_m3": 25.0, "renovacao_pct_dia": 0.5, "energia_kwh_kg": 3.00},
}


@dataclass
class TecnicoConfig:
    # Bloco 1 — infraestrutura
    sistema_produtivo: str = "semi-intensivo"
    area_lamina_m2: float = 40000.0
    profundidade_media_m: float = 1.4
    numero_tanques: int = 32
    tanques_ativos: int = 30
    capacidade_suporte_kg_m3: Optional[float] = None   # None => referência do sistema
    produtividade_esperada_kg_m2_ciclo: float = 1.3     # restrição por área (0 = ignora)
    # Bloco 2 — parâmetros biológicos
    especie: str = "Tilápia-do-nilo (Oreochromis niloticus)"
    peso_inicial_g: float = 30.0
    peso_final_g: float = 800.0
    ciclo_dias: int = 180
    ciclos_ano: float = 1.8
    alevinos_por_ciclo: float = 0.0                     # 0 => dimensionado pelo modelo
    desempenho_crescimento_pct: float = 100.0           # peso atingido / peso-alvo
    # Bloco 3 — eficiência de insumos
    fcr: float = 1.50
    mortalidade_pct: float = 10.0
    fcr_referencia: float = 1.40                        # benchmark Lean-Green
    mortalidade_referencia_pct: float = 5.0

    @property
    def volume_util_m3(self) -> float:
        return self.area_lamina_m2 * self.profundidade_media_m


# ---------------------------------------------------------------------------
# Dimensão econômica — CAPEX, OPEX, receitas e parâmetros financeiros
# ---------------------------------------------------------------------------

def _capex_padrao() -> Dict[str, float]:
    return {
        "Escavação e construção de viveiros": 400000.0,
        "Sistema de abastecimento e drenagem": 70000.0,
        "Aeradores e quadro elétrico": 90000.0,
        "Bombas e tubulações": 35000.0,
        "Galpão, depósito de ração e equipamentos": 80000.0,
        "Lagoa de sedimentação / tratamento de efluentes": 45000.0,
        "Licenciamento, outorga e projetos": 25000.0,
    }


# Faixas de exatidão típicas (limite inferior, limite superior) por classe de
# estimativa AACE 18R-97, usadas para a incerteza do CAPEX.
FAIXAS_AACE = {5: (-0.30, 0.50), 4: (-0.20, 0.30), 3: (-0.15, 0.20), 2: (-0.10, 0.15), 1: (-0.05, 0.10)}


@dataclass
class EconomicoConfig:
    tipo_organizacao: str = "coop_agro"          # empresa | coop_trabalho | coop_agro
    capex_itens: Dict[str, float] = field(default_factory=_capex_padrao)
    classe_estimativa_aace: int = 5
    capital_giro: float = 60000.0
    vida_util_anos: float = 15.0                 # depreciação linear
    valor_residual_pct: float = 30.0             # % do CAPEX recuperável no fim
    # Receitas
    preco_venda_kg: float = 10.50
    mix_produtos: List[Dict[str, Any]] = field(default_factory=list)  # [{nome, participacao_pct, preco_kg}]
    receita_servicos_mes: float = 0.0
    # OPEX variável
    custo_racao_kg: float = 3.10
    custo_alevino_milheiro: float = 280.0
    tarifa_energia_kwh: float = 0.75
    outros_custos_variaveis_kg: float = 0.35     # despesca, gelo, transporte, embalagem
    # OPEX fixo (mensal)
    mao_obra_mes: float = 12000.0                # salários / retiradas
    encargos_mao_obra_pct: float = 20.0
    assistencia_tecnica_mes: float = 2000.0
    administrativo_mes: float = 1500.0
    manutencao_capex_pct_aa: float = 2.0
    custos_fixos_fator: float = 1.0              # multiplicador para incerteza
    # Tributos
    taxa_impostos_faturamento_pct: float = 2.3   # ex.: contribuições sobre a receita
    taxa_impostos_lucro_pct: float = 0.0         # cooperativas: ato cooperativo
    impostos_configurados: List[Dict[str, Any]] = field(default_factory=list)
    # Parâmetros financeiros
    tma_aa_pct: float = 10.0
    horizonte_anos: int = 10
    crescimento_preco_aa_pct: float = 0.0
    crescimento_custos_aa_pct: float = 0.0
    # Rampa de capacidade (aprendizado da equipe)
    capacidade_inicial_pct: float = 70.0
    meses_rampa: int = 12
    # Cooperativas
    cooperados_trabalhadores: int = 6
    cooperados_fornecedores: int = 0
    perc_sobras_trabalhadores: float = 100.0
    capex_fator: float = 1.0                     # multiplicador para incerteza (AACE)
    # Fomento não reembolsável (fundos mitigatórios/compensatórios, editais)
    fomento_nao_reembolsavel_pct: float = 0.0    # % do CAPEX doado ao beneficiário

    @property
    def capex_total(self) -> float:
        return float(sum(self.capex_itens.values()))


# ---------------------------------------------------------------------------
# Dimensão ambiental — gestão hídrica, energética, efluentes e conformidade
# ---------------------------------------------------------------------------

# Checklist normativo do Score de Conformidade Ambiental (0–10). Os itens
# marcados como eliminatórios bloqueiam a classificação "VIÁVEL".
CHECKLIST_AMBIENTAL = {
    "licenca_ambiental": ("Licença ambiental do empreendimento aquícola (Resolução CONAMA nº 413/2009)", 2.0, True),
    "outorga_agua": ("Outorga de direito de uso de recursos hídricos (Lei nº 9.433/1997)", 1.5, True),
    "respeita_app": ("Instalações fora da Área de Preservação Permanente (Lei nº 12.651/2012)", 1.5, True),
    "tratamento_efluentes": ("Tratamento de efluentes antes do lançamento (CONAMA nº 357/2005 e 430/2011)", 1.5, False),
    "monitoramento_agua": ("Monitoramento periódico da qualidade da água", 1.0, False),
    "car": ("Inscrição no Cadastro Ambiental Rural", 0.5, False),
    "gestao_residuos": ("Plano de gestão de resíduos sólidos e lodo", 0.75, False),
    "prevencao_escape": ("Barreiras e plano de prevenção de escape de espécie exótica", 0.75, False),
    "energia_renovavel": ("Uso de energia renovável ou eficiência energética", 0.5, False),
}

FATORES_EMISSAO_ENERGIA = {  # kgCO2e/kWh — atualizar ao ano-base do estudo
    "rede": 0.0817,
    "solar": 0.0,
    "biomassa": 0.0,
}


@dataclass
class AmbientalConfig:
    # Gestão hídrica
    renovacao_pct_dia: Optional[float] = None    # None => referência do sistema
    esvaziamento_por_ciclo: bool = True
    evaporacao_mm_dia: float = 4.0
    infiltracao_mm_dia: float = 2.0
    fracao_retorno_perdida: float = 0.0           # água devolvida a outra bacia
    # Balanço de nutrientes (pegada hídrica cinza — Hoekstra)
    proteina_racao_pct: float = 32.0
    fosforo_racao_pct: float = 1.0
    nitrogenio_peixe_pct: float = 2.6
    fosforo_peixe_pct: float = 0.7
    remocao_n_tratamento_pct: float = 40.0
    remocao_p_tratamento_pct: float = 60.0
    n_max_mg_l: float = 2.18                      # CONAMA 357/2005, classe 2 lótico
    p_max_mg_l: float = 0.10                      # CONAMA 357/2005, classe 2 lótico
    n_natural_mg_l: float = 0.50
    p_natural_mg_l: float = 0.02
    # Gestão energética e emissões
    energia_kwh_kg: Optional[float] = None        # None => referência do sistema
    fonte_energia: str = "rede"                   # rede | solar | biomassa | mista
    fracao_renovavel_mista: float = 0.0
    fator_emissao_racao_kgco2_kg: float = 1.1
    diesel_l_ano: float = 600.0
    fator_emissao_diesel_kgco2_l: float = 2.6
    # Resíduos
    residuos_kg_por_kg: float = 0.04
    residuos_reaproveitados_pct: float = 50.0
    # Conformidade
    distancia_app_m: float = 50.0
    faixa_app_minima_m: float = 30.0
    checklist: Dict[str, bool] = field(default_factory=lambda: {
        "licenca_ambiental": True, "outorga_agua": True, "respeita_app": True,
        "tratamento_efluentes": True, "monitoramento_agua": True, "car": True,
        "gestao_residuos": False, "prevencao_escape": True, "energia_renovavel": False,
    })
    # Referências para normalização / Lean-Green
    energia_referencia_kwh_kg: float = 0.40


# ---------------------------------------------------------------------------
# Dimensão social — Radar Social, LSO, RVL e ODS
# ---------------------------------------------------------------------------

RELACAO_COMUNIDADE = {"inexistente": 0.0, "parcial": 0.5, "consolidado": 1.0}


@dataclass
class SocialConfig:
    empregos_diretos: int = 6
    empregos_indiretos: int = 10
    mao_obra_local_pct: float = 100.0
    compras_locais_pct: float = 25.0            # % das compras de insumos feitas localmente
    salario_minimo: float = 1518.0
    capacitacao_horas_pessoa_ano: float = 40.0
    conformidade_nr_pct: float = 85.0
    relacao_comunidade: str = "parcial"          # inexistente | parcial | consolidado
    canais_formais_comunicacao: bool = True
    reunioes_comunidade_ano: int = 6
    conflitos_registrados_ano: int = 1
    mulheres_pct: float = 40.0
    jovens_pct: float = 20.0
    publico_vulneravel: bool = True              # pescadores artesanais / PGTR


@dataclass
class GovernancaConfig:
    """Checklist do pilar G (ESG)."""
    itens: Dict[str, bool] = field(default_factory=lambda: {
        "estatuto_regimento": True,
        "assembleias_regulares": True,
        "conselho_fiscal": True,
        "prestacao_contas_publica": False,
        "plano_negocios_aprovado": True,
        "registro_auditavel_premissas": True,
        "canal_denuncia_ouvidoria": False,
    })


# ---------------------------------------------------------------------------
# Ponderação, decisão e incerteza
# ---------------------------------------------------------------------------

KPIS_POR_DIMENSAO = {
    "tecnico": ["oee", "fcr", "produtividade"],
    "economico": ["vpl", "roi", "margem_seguranca"],
    "ambiental": ["ph_azul", "ph_cinza", "ecoeficiencia", "intensidade_carbono", "conformidade"],
    "social": ["lso", "rvl", "radar_social"],
}


@dataclass
class PesosConfig:
    metodo: str = "preset"                        # preset | likert | ahp
    tecnico: float = 0.25
    economico: float = 0.35
    ambiental: float = 0.20
    social: float = 0.20
    kpi: Dict[str, float] = field(default_factory=dict)  # vazio => pesos iguais na dimensão
    likert_dimensoes: Dict[str, Dict[str, int]] = field(default_factory=dict)  # {avaliador: {dim: 1..5}}
    likert_kpis: Dict[str, Dict[str, int]] = field(default_factory=dict)        # {avaliador: {kpi: 1..5}}
    ahp_matriz: Optional[List[List[float]]] = None                               # ordem: tec, eco, amb, soc

    def vetor(self) -> Dict[str, float]:
        v = np.array([self.tecnico, self.economico, self.ambiental, self.social], dtype=float)
        if np.any(v < 0) or v.sum() <= 0:
            raise ValueError("Pesos dimensionais inválidos")
        v = v / v.sum()
        return dict(zip(["tecnico", "economico", "ambiental", "social"], v.tolist()))


@dataclass
class DecisaoConfig:
    limiar_indice_viavel: float = 0.60
    limiar_indice_ressalvas: float = 0.50
    prob_vpl_positivo_minima: float = 0.70
    lso_minimo: float = 40.0
    aplicar_vetos: bool = True


def _distribuicoes_padrao() -> Dict[str, Distribuicao]:
    return {
        "economico.preco_venda_kg": Distribuicao("triangular", 9.00, 10.50, 11.50),
        "economico.custo_racao_kg": Distribuicao("triangular", 2.80, 3.10, 3.70),
        "economico.custo_alevino_milheiro": Distribuicao("triangular", 230.0, 280.0, 350.0),
        "economico.tarifa_energia_kwh": Distribuicao("triangular", 0.65, 0.75, 0.95),
        "economico.custos_fixos_fator": Distribuicao("triangular", 0.95, 1.00, 1.15),
        "tecnico.fcr": Distribuicao("triangular", 1.30, 1.50, 1.85),
        "tecnico.mortalidade_pct": Distribuicao("triangular", 5.0, 10.0, 20.0),
        "tecnico.desempenho_crescimento_pct": Distribuicao("triangular", 85.0, 100.0, 105.0),
    }


@dataclass
class MonteCarloConfig:
    iteracoes: int = 10000
    seed: int = 20260612
    distribuicoes: Dict[str, Distribuicao] = field(default_factory=_distribuicoes_padrao)
    incluir_capex_aace: bool = True               # adiciona economico.capex_fator pela classe AACE


@dataclass
class EVTEASConfig:
    projeto: str = "Projeto de Piscicultura — EVTEAS-Py"
    autor: str = "Rafael Alves Bastos"
    versao: str = VERSAO
    tecnico: TecnicoConfig = field(default_factory=TecnicoConfig)
    economico: EconomicoConfig = field(default_factory=EconomicoConfig)
    ambiental: AmbientalConfig = field(default_factory=AmbientalConfig)
    social: SocialConfig = field(default_factory=SocialConfig)
    governanca: GovernancaConfig = field(default_factory=GovernancaConfig)
    pesos: PesosConfig = field(default_factory=PesosConfig)
    decisao: DecisaoConfig = field(default_factory=DecisaoConfig)
    monte_carlo: MonteCarloConfig = field(default_factory=MonteCarloConfig)
    fontes: Dict[str, Fonte] = field(default_factory=dict)  # "bloco.campo" -> Fonte


# ---------------------------------------------------------------------------
# Acesso por caminho ("bloco.campo") e (de)serialização
# ---------------------------------------------------------------------------

def obter_por_caminho(cfg: EVTEASConfig, caminho: str) -> Any:
    obj: Any = cfg
    for parte in caminho.split("."):
        obj = getattr(obj, parte)
    return obj


def definir_por_caminho(cfg: EVTEASConfig, caminho: str, valor: Any) -> None:
    partes = caminho.split(".")
    obj: Any = cfg
    for parte in partes[:-1]:
        obj = getattr(obj, parte)
    if not hasattr(obj, partes[-1]):
        raise AttributeError(f"Parâmetro inexistente: {caminho}")
    setattr(obj, partes[-1], valor)


def caminhos_numericos(cfg: EVTEASConfig) -> List[str]:
    """Lista os parâmetros numéricos escalares da configuração."""
    saida = []
    for bloco in ("tecnico", "economico", "ambiental", "social"):
        obj = getattr(cfg, bloco)
        for f in fields(obj):
            v = getattr(obj, f.name)
            if isinstance(v, (int, float)) and not isinstance(v, bool):
                saida.append(f"{bloco}.{f.name}")
    return saida


def config_para_dict(cfg: EVTEASConfig) -> Dict[str, Any]:
    return asdict(cfg)


def _construir(tipo, dados):
    if not is_dataclass(tipo) or not isinstance(dados, dict):
        return dados
    kwargs = {}
    for f in fields(tipo):
        if f.name not in dados:
            continue
        v = dados[f.name]
        if f.name == "distribuicoes":
            v = {k: Distribuicao(**d) for k, d in v.items()}
        elif f.name == "fontes":
            v = {k: Fonte(**d) for k, d in v.items()}
        elif isinstance(f.type, str) and f.type.endswith("Config") and isinstance(v, dict):
            v = _construir(globals()[f.type], v)
        kwargs[f.name] = v
    return tipo(**kwargs)


def config_de_dict(dados: Dict[str, Any]) -> EVTEASConfig:
    return _construir(EVTEASConfig, dados)

## 2. Engenharia econômica vetorizada (VPL, TIR com diagnóstico, payback)

In [ ]:
"""Utilitários numéricos e de engenharia econômica (vetorizados).

Todas as funções aceitam fluxos com forma (M+1,) ou (n, M+1), em que n é o
número de iterações de Monte Carlo. Assim o modelo determinístico e o
estocástico usam exatamente as mesmas fórmulas (verificação de consistência).
"""

from typing import Dict, Tuple

import numpy as np


def safe_float(x, default: float = 0.0) -> float:
    try:
        x = float(x)
        return x if np.isfinite(x) else default
    except (TypeError, ValueError):
        return default


def pct(x):
    return np.asarray(x, dtype=float) / 100.0


def safe_div(a, b, default=0.0):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        out = np.where(np.abs(b) < 1e-15, default, a / np.where(np.abs(b) < 1e-15, 1.0, b))
    return out if out.ndim else float(out)


def clamp(x, lo=0.0, hi=1.0):
    out = np.clip(np.nan_to_num(np.asarray(x, dtype=float), nan=lo), lo, hi)
    return out if out.ndim else float(out)


def taxa_anual_para_mensal(taxa_aa) -> np.ndarray:
    return (1 + np.asarray(taxa_aa, dtype=float)) ** (1 / 12) - 1


def taxa_mensal_para_anual(taxa_am) -> np.ndarray:
    return (1 + np.asarray(taxa_am, dtype=float)) ** 12 - 1


def _2d(fluxos) -> np.ndarray:
    f = np.asarray(fluxos, dtype=float)
    return f[None, :] if f.ndim == 1 else f


def vpl(taxa, fluxos):
    """Valor presente líquido; o índice 0 é a data focal (não descontado)."""
    f = _2d(fluxos)
    taxa = np.asarray(taxa, dtype=float).reshape(-1, 1)
    t = np.arange(f.shape[1])[None, :]
    with np.errstate(over="ignore", invalid="ignore"):
        out = (f / (1 + taxa) ** t).sum(axis=1)
    return float(out[0]) if np.ndim(fluxos) == 1 else out


def trocas_de_sinal(fluxos) -> np.ndarray:
    f = _2d(fluxos)
    s = np.sign(f)
    out = np.zeros(f.shape[0], dtype=int)
    for i in range(f.shape[0]):
        nz = s[i][s[i] != 0]
        out[i] = int(np.sum(nz[1:] != nz[:-1])) if nz.size > 1 else 0
    return out


def tir(fluxos, lo: float = -0.90, hi: float = 1.0, pontos: int = 160,
        iteracoes: int = 100) -> Tuple[np.ndarray, np.ndarray]:
    """TIR por varredura de malha + bisseção vetorizada, com diagnóstico explícito.

    Retorna (taxa por período, diagnóstico). A taxa é ``nan`` quando não há raiz
    no intervalo: nunca é forçada ao limite numérico (corrige a leitura de
    -0,95 relatada na Rev. 186). Diagnósticos: ``convencional`` (uma raiz e uma
    troca de sinal no fluxo), ``multiplas`` (mais de uma raiz na malha),
    ``multiplas_possiveis`` (uma raiz, mas mais de uma troca de sinal —
    regra de Descartes) e ``sem_solucao`` (o VPL não muda de sinal).
    """
    f = _2d(fluxos)
    n = f.shape[0]
    malha = np.unique(np.concatenate([np.linspace(lo, -0.5, 12), np.linspace(-0.5, hi, pontos)]))
    valores = np.vstack([vpl(np.full(n, r), f) for r in malha]).T          # (n, pontos)
    troca = (valores[:, :-1] * valores[:, 1:]) <= 0
    n_raizes = troca.sum(axis=1)
    existe = n_raizes > 0
    k = np.where(existe, troca.argmax(axis=1), 0)
    a, b = malha[k].astype(float), malha[k + 1].astype(float)
    fa = valores[np.arange(n), k]
    for _ in range(iteracoes):
        m = (a + b) / 2
        fm = vpl(m, f)
        esquerda = (fa * fm) <= 0
        b = np.where(esquerda, m, b)
        a = np.where(esquerda, a, m)
        fa = np.where(esquerda, fa, fm)
    raiz = np.where(existe, (a + b) / 2, np.nan)
    trocas = trocas_de_sinal(f)
    diag = np.where(~existe, "sem_solucao",
                    np.where(n_raizes > 1, "multiplas",
                             np.where(trocas > 1, "multiplas_possiveis", "convencional")))
    return raiz, diag


def payback(fluxos, taxa=None) -> np.ndarray:
    """Payback (simples se ``taxa`` é None, descontado caso contrário), em períodos.

    Retorna ``nan`` quando o investimento não é recuperado no horizonte.
    Interpolação linear dentro do período de recuperação.
    """
    f = _2d(fluxos)
    if taxa is not None:
        t = np.arange(f.shape[1])[None, :]
        f = f / (1 + np.asarray(taxa, dtype=float).reshape(-1, 1)) ** t
    acum = np.cumsum(f, axis=1)
    out = np.full(f.shape[0], np.nan)
    for i in range(f.shape[0]):
        idx = np.where((acum[i, 1:] >= 0) & (acum[i, :-1] < 0))[0]
        if idx.size:
            k = idx[0] + 1
            out[i] = (k - 1) + safe_div(-acum[i, k - 1], f[i, k], 0.0)
    return out


def percentis(valores, ps=(1, 5, 10, 25, 50, 75, 90, 95, 99)) -> Dict[str, float]:
    x = np.asarray(valores, dtype=float)
    x = x[np.isfinite(x)]
    if not x.size:
        return {f"p{p}": float("nan") for p in ps}
    return {f"p{p}": float(np.percentile(x, p)) for p in ps}

## 3. Núcleo de cálculo: dimensões técnica, econômica, ambiental, social e Lean-Green

In [ ]:
"""Núcleo de cálculo vetorizado das quatro dimensões do EVTEAS.

Cada função recebe um ``Contexto``, que devolve qualquer parâmetro numérico
como vetor de tamanho n. Com n = 1 o modelo é determinístico; com n = 10.000
ele é a própria simulação de Monte Carlo. Não existe um "modelo simplificado"
para a camada estocástica: o mesmo código produz os dois resultados.
"""

import math
from typing import Any, Dict, Optional

import numpy as np
import pandas as pd



class Contexto:
    """Acesso vetorizado aos parâmetros, com substituições (overrides) por caminho."""

    def __init__(self, cfg: EVTEASConfig, overrides: Optional[Dict[str, np.ndarray]] = None, n: int = 1):
        self.cfg = cfg
        self.overrides = overrides or {}
        self.n = int(n)
        for k, v in self.overrides.items():
            if np.ndim(v) and len(v) != self.n:
                raise ValueError(f"Override '{k}' com tamanho {len(v)} != n={self.n}")

    def v(self, caminho: str) -> np.ndarray:
        if caminho in self.overrides:
            return np.broadcast_to(np.asarray(self.overrides[caminho], dtype=float), (self.n,)).astype(float)
        valor = obter_por_caminho(self.cfg, caminho)
        return np.full(self.n, float(valor), dtype=float)


def referencia_sistema(cfg: EVTEASConfig, chave: str) -> float:
    sistema = cfg.tecnico.sistema_produtivo
    if sistema not in SISTEMAS_PRODUTIVOS:
        raise ValueError(f"Sistema produtivo desconhecido: {sistema}")
    return float(SISTEMAS_PRODUTIVOS[sistema][chave])


def capacidade_suporte(cfg: EVTEASConfig) -> float:
    c = cfg.tecnico.capacidade_suporte_kg_m3
    return float(c) if c is not None else referencia_sistema(cfg, "capacidade_suporte_kg_m3")


def energia_kwh_kg(cfg: EVTEASConfig) -> float:
    e = cfg.ambiental.energia_kwh_kg
    return float(e) if e is not None else referencia_sistema(cfg, "energia_kwh_kg")


def renovacao_pct_dia(cfg: EVTEASConfig) -> float:
    r = cfg.ambiental.renovacao_pct_dia
    return float(r) if r is not None else referencia_sistema(cfg, "renovacao_pct_dia")


# ---------------------------------------------------------------------------
# 1. Dimensão técnica
# ---------------------------------------------------------------------------

def calcular_tecnico(ctx: Contexto) -> Dict[str, Any]:
    t = ctx.cfg.tecnico
    vol = t.area_lamina_m2 * t.profundidade_media_m
    disponibilidade = safe_div(t.tanques_ativos, t.numero_tanques, 0.0)
    cap = capacidade_suporte(ctx.cfg)

    mort = np.clip(ctx.v("tecnico.mortalidade_pct"), 0, 100)
    sobrevivencia = 1 - mort / 100
    desempenho = np.clip(ctx.v("tecnico.desempenho_crescimento_pct"), 0, None) / 100
    fcr = np.maximum(ctx.v("tecnico.fcr"), 0)
    ciclos = max(float(t.ciclos_ano), 0.0)

    # Restrição dupla da biomassa: capacidade de suporte (volume) e produtividade (área)
    limite_volume = vol * cap * disponibilidade
    limite_area = (t.area_lamina_m2 * t.produtividade_esperada_kg_m2_ciclo * disponibilidade
                   if t.produtividade_esperada_kg_m2_ciclo > 0 else np.inf)
    biomassa_alvo = min(limite_volume, limite_area)
    restricao = "capacidade de suporte (volume)" if limite_volume <= limite_area else "produtividade por área"

    peso_final_kg = t.peso_final_g / 1000
    sobrevivencia_planejada = 1 - t.mortalidade_pct / 100
    alevinos = (t.alevinos_por_ciclo if t.alevinos_por_ciclo > 0
                else safe_div(biomassa_alvo, peso_final_kg * sobrevivencia_planejada, 0.0))

    # A estocagem é decisão de projeto; a mortalidade e o crescimento são incertos.
    biomassa_despesca = np.minimum(alevinos * sobrevivencia * peso_final_kg * desempenho, limite_volume)
    biomassa_inicial = alevinos * t.peso_inicial_g / 1000
    ganho_ciclo = np.maximum(biomassa_despesca - biomassa_inicial, 0)
    racao_ciclo = ganho_ciclo * fcr

    producao_ano = biomassa_despesca * ciclos
    performance = np.minimum(desempenho, 1.0)
    oee = disponibilidade * performance * sobrevivencia     # Quadro 5: Qualidade = sobrevivência

    potencial_kg_m2_ano = cap * t.profundidade_media_m * min(ciclos, 365 / max(t.ciclo_dias, 1))
    produtividade_kg_m2_ano = safe_div(producao_ano, t.area_lamina_m2, 0.0)

    return {
        "especie": t.especie,
        "sistema_produtivo": t.sistema_produtivo,
        "volume_util_m3": vol,
        "capacidade_suporte_kg_m3": cap,
        "restricao_biomassa": restricao,
        "disponibilidade": np.full(ctx.n, disponibilidade),
        "performance": performance,
        "qualidade": sobrevivencia,
        "oee": oee,
        "sobrevivencia": sobrevivencia,
        "fcr": fcr,
        "alevinos_ciclo": np.full(ctx.n, alevinos),
        "alevinos_ano": np.full(ctx.n, alevinos * ciclos),
        "densidade_estocagem_peixes_m3": np.full(ctx.n, safe_div(alevinos, vol * disponibilidade, 0.0)),
        "biomassa_inicial_ciclo_kg": np.full(ctx.n, biomassa_inicial),
        "biomassa_despesca_ciclo_kg": biomassa_despesca,
        "ganho_biomassa_ciclo_kg": ganho_ciclo,
        "ganho_biomassa_ano_kg": ganho_ciclo * ciclos,
        "racao_ciclo_kg": racao_ciclo,
        "racao_ano_kg": racao_ciclo * ciclos,
        "racao_por_kg_produzido": safe_div(racao_ciclo, biomassa_despesca, 0.0),
        "producao_kg_ano": producao_ano,
        "producao_kg_mes": producao_ano / 12,
        "produtividade_kg_m2_ano": produtividade_kg_m2_ano,
        "produtividade_volumetrica_kg_m3_ciclo": safe_div(biomassa_despesca, vol, 0.0),
        "produtividade_relativa": clamp(safe_div(produtividade_kg_m2_ano, potencial_kg_m2_ano, 0.0), 0, 1),
        "ciclos_ano": ciclos,
    }


def curva_crescimento(cfg: EVTEASConfig, passo_dias: int = 7) -> pd.DataFrame:
    """Curva de crescimento e de arraçoamento de um ciclo (modelo de TCE).

    Peso: W(d) = W0 · exp(TCE · d), TCE = ln(Wf/W0)/ciclo. Mortalidade
    distribuída exponencialmente ao longo do ciclo. A ração diária é
    proporcional ao ganho de biomassa e calibrada para reproduzir o FCR.
    """
    t = cfg.tecnico
    ctx = Contexto(cfg)
    tec = calcular_tecnico(ctx)
    n0 = float(tec["alevinos_ciclo"][0])
    w0, wf = t.peso_inicial_g / 1000, t.peso_final_g / 1000 * t.desempenho_crescimento_pct / 100
    dias = np.arange(0, t.ciclo_dias + 1)
    tce = math.log(wf / w0) / t.ciclo_dias if wf > w0 > 0 else 0.0
    peso = w0 * np.exp(tce * dias)
    m_diaria = 1 - (1 - t.mortalidade_pct / 100) ** (1 / t.ciclo_dias)
    vivos = n0 * (1 - m_diaria) ** dias
    biomassa = vivos * peso
    ganho = np.diff(biomassa, prepend=biomassa[0]).clip(min=0)
    racao_dia = ganho * t.fcr
    df = pd.DataFrame({
        "Dia": dias, "Peso médio (g)": peso * 1000, "Peixes vivos": vivos,
        "Biomassa (kg)": biomassa, "Ração do dia (kg)": racao_dia,
        "Ração acumulada (kg)": np.cumsum(racao_dia),
        "TCE (%/dia)": tce * 100,
    })
    if passo_dias <= 1:
        return df
    return df[(df["Dia"] % passo_dias == 0) | (df["Dia"] == t.ciclo_dias)].reset_index(drop=True)


# ---------------------------------------------------------------------------
# 2. Dimensão econômica
# ---------------------------------------------------------------------------

def preco_medio(cfg: EVTEASConfig) -> float:
    mix = cfg.economico.mix_produtos
    if not mix:
        return float(cfg.economico.preco_venda_kg)
    part = np.array([float(p.get("participacao_pct", 0)) for p in mix])
    preco = np.array([float(p.get("preco_kg", cfg.economico.preco_venda_kg)) for p in mix])
    if part.sum() <= 0:
        raise ValueError("Participações do mix de produtos devem somar mais que zero")
    return float((part / part.sum() * preco).sum())


def aliquotas_configuradas(cfg: EVTEASConfig) -> Dict[str, float]:
    """Consolida alíquotas individuais (ICMS, PIS, COFINS, IRPJ, CSLL, INSS...) por base."""
    base = {"faturamento": pct(cfg.economico.taxa_impostos_faturamento_pct).item(),
            "lucro": pct(cfg.economico.taxa_impostos_lucro_pct).item(),
            "folha": pct(cfg.economico.encargos_mao_obra_pct).item()}
    for imp in cfg.economico.impostos_configurados:
        b = imp.get("base", "faturamento")
        if b not in base:
            raise ValueError(f"Base tributária desconhecida: {b}")
        base[b] += float(imp.get("aliquota_pct", 0)) / 100
    return base


def meses_ate_despesca(cfg: EVTEASConfig) -> int:
    return int(math.ceil(cfg.tecnico.ciclo_dias / 30.4375))


def calcular_economico(ctx: Contexto, tec: Dict[str, Any]) -> Dict[str, Any]:
    cfg, e = ctx.cfg, ctx.cfg.economico
    n = ctx.n
    H = int(max(e.horizonte_anos, 1))
    M = H * 12
    meses = np.arange(1, M + 1)
    ano = np.ceil(meses / 12).astype(int)
    L = meses_ate_despesca(cfg)
    aliq = aliquotas_configuradas(cfg)

    # Rampa de capacidade (aprendizado) aplicada à produção em engorda
    cap0 = e.capacidade_inicial_pct / 100
    rampa = cap0 + (1 - cap0) * np.minimum((meses - 1) / max(e.meses_rampa, 1), 1.0) if e.meses_rampa > 0 else np.ones(M)
    rampa = rampa[None, :]

    fator_preco = ((1 + e.crescimento_preco_aa_pct / 100) ** (ano - 1))[None, :]
    fator_custo = ((1 + e.crescimento_custos_aa_pct / 100) ** (ano - 1))[None, :]

    prod_mes = tec["producao_kg_mes"][:, None]
    em_engorda = prod_mes * rampa                                   # kg/mês em crescimento
    vendida = np.zeros((n, M))
    vendida[:, L:] = em_engorda[:, : M - L]                        # despesca após o 1º ciclo

    escala_preco = safe_div(ctx.v("economico.preco_venda_kg"), e.preco_venda_kg, 1.0)
    preco = preco_medio(cfg) * escala_preco[:, None] * fator_preco
    receita_produtos = vendida * preco
    receita_servicos = np.where(meses > L, e.receita_servicos_mes, 0.0)[None, :] * fator_preco * np.ones((n, 1))
    receita = receita_produtos + receita_servicos

    custo_racao = em_engorda * tec["racao_por_kg_produzido"][:, None] * ctx.v("economico.custo_racao_kg")[:, None] * fator_custo
    alevinos_mes = tec["alevinos_ano"][:, None] / 12 * rampa
    custo_alevinos = alevinos_mes * ctx.v("economico.custo_alevino_milheiro")[:, None] / 1000 * fator_custo
    kwh_kg = energia_kwh_kg(cfg)
    custo_energia = em_engorda * kwh_kg * ctx.v("economico.tarifa_energia_kwh")[:, None] * fator_custo
    outros_var = vendida * e.outros_custos_variaveis_kg * fator_custo
    custos_variaveis = custo_racao + custo_alevinos + custo_energia + outros_var

    capex = e.capex_total * ctx.v("economico.capex_fator")
    fixos_fator = ctx.v("economico.custos_fixos_fator")[:, None]
    pessoal = e.mao_obra_mes * (1 + aliq["folha"])
    manutencao = capex[:, None] * e.manutencao_capex_pct_aa / 100 / 12
    custos_fixos = (pessoal + e.assistencia_tecnica_mes + e.administrativo_mes + manutencao) * fixos_fator * fator_custo

    vida_meses = max(e.vida_util_anos, 0.0) * 12
    depreciacao = np.where(meses <= vida_meses, 1.0, 0.0)[None, :] * safe_div(capex, vida_meses, 0.0)[:, None]

    impostos_fat = receita * aliq["faturamento"]
    receita_liquida = receita - impostos_fat
    lucro_bruto = receita_liquida - custos_variaveis
    ebitda = lucro_bruto - custos_fixos
    lair = ebitda - depreciacao
    impostos_lucro = np.maximum(lair, 0) * aliq["lucro"]
    lucro_liquido = lair - impostos_lucro
    fco = lucro_liquido + depreciacao

    investimento = capex + e.capital_giro
    residual = capex * e.valor_residual_pct / 100
    estoque_final = (custo_racao + custo_alevinos + custo_energia)[:, M - L:].sum(axis=1) if L > 0 else np.zeros(n)
    terminal = residual + e.capital_giro + estoque_final

    fluxo = np.zeros((n, M + 1))
    fluxo[:, 0] = -investimento
    fluxo[:, 1:] = fco
    fluxo[:, -1] += terminal

    tma_am = float(taxa_anual_para_mensal(e.tma_aa_pct / 100))
    vpl_v = vpl(np.full(n, tma_am), fluxo)
    tir_am, diag_tir = tir(fluxo)
    tir_aa = taxa_mensal_para_anual(tir_am)
    pb_s = payback(fluxo)
    pb_d = payback(fluxo, np.full(n, tma_am))
    il = safe_div(vpl_v + investimento, investimento, np.nan)

    # Perspectiva do beneficiário: parte do CAPEX aportada por fomento não reembolsável
    fomento = capex * e.fomento_nao_reembolsavel_pct / 100
    fluxo_benef = fluxo.copy()
    fluxo_benef[:, 0] += fomento
    vpl_benef = vpl(np.full(n, tma_am), fluxo_benef)
    tir_benef_am, _ = tir(fluxo_benef)

    # Ano de regime: primeiro ano inteiro com capacidade plena e vendas
    ano_regime = int(min(math.ceil((max(e.meses_rampa, 0) + L) / 12) + 1, H))
    sel = ano == ano_regime

    def anual(x):
        return x[:, sel].sum(axis=1)

    rec_r, var_r, fix_r = anual(receita), anual(custos_variaveis), anual(custos_fixos)
    imp_r, dep_r, ll_r = anual(impostos_fat), anual(depreciacao), anual(lucro_liquido)
    kg_r = anual(vendida)
    imc = safe_div(rec_r - imp_r - var_r, rec_r, 0.0)
    pe_receita = np.where(imc > 0, safe_div(fix_r + dep_r, np.where(imc > 0, imc, 1.0), np.nan), np.nan)
    margem_seg = np.where(np.isfinite(pe_receita), safe_div(rec_r - pe_receita, rec_r, np.nan), -1.0)
    valor_adicionado = rec_r - (anual(custo_racao) + anual(custo_alevinos) + anual(custo_energia) + anual(outros_var))

    return {
        "meses_ate_despesca": L,
        "ano_regime": ano_regime,
        "aliquotas": aliq,
        "preco_medio_kg": preco[:, 0],
        "capex": capex,
        "investimento_total": investimento,
        "fluxo_caixa": fluxo,
        "tma_mensal": tma_am,
        "vpl": vpl_v,
        "tir_mensal": tir_am,
        "tir_anual": tir_aa,
        "diagnostico_tir": diag_tir,
        "payback_simples_meses": pb_s,
        "payback_descontado_meses": pb_d,
        "indice_lucratividade": il,
        "fomento_nao_reembolsavel": fomento,
        "fluxo_caixa_beneficiario": fluxo_benef,
        "vpl_beneficiario": vpl_benef,
        "tir_anual_beneficiario": taxa_mensal_para_anual(tir_benef_am),
        "payback_descontado_beneficiario_meses": payback(fluxo_benef, np.full(n, tma_am)),
        "receita_regime": rec_r,
        "custos_variaveis_regime": var_r,
        "custos_fixos_regime": fix_r,
        "impostos_faturamento_regime": imp_r,
        "depreciacao_regime": dep_r,
        "lucro_liquido_regime": ll_r,
        "ebitda_regime": anual(ebitda),
        "kg_vendidos_regime": kg_r,
        "custo_racao_regime": anual(custo_racao),
        "custo_alevinos_regime": anual(custo_alevinos),
        "custo_energia_regime": anual(custo_energia),
        "outros_variaveis_regime": anual(outros_var),
        "pessoal_regime": anual(pessoal * fixos_fator * fator_custo * np.ones((n, M))),
        "valor_adicionado_regime": valor_adicionado,
        "indice_margem_contribuicao": imc,
        "ponto_equilibrio_receita_ano": pe_receita,
        "ponto_equilibrio_kg_ano": safe_div(pe_receita, safe_div(rec_r, kg_r, np.nan), np.nan),
        "margem_seguranca": margem_seg,
        "roi_anual_regime": safe_div(ll_r, investimento, np.nan),
        "roi_horizonte": safe_div(lucro_liquido.sum(axis=1), investimento, np.nan),
        "margem_liquida_regime": safe_div(ll_r, rec_r, np.nan),
        "custo_total_kg": safe_div(var_r + fix_r + dep_r + imp_r, kg_r, np.nan),
        "custo_operacional_efetivo_kg": safe_div(var_r + fix_r, kg_r, np.nan),
        "receita_total": receita.sum(axis=1),
        "lucro_liquido_total": lucro_liquido.sum(axis=1),
        "_series": {  # séries mensais (n, M) para a DRE
            "meses": meses, "ano": ano, "capacidade": np.broadcast_to(rampa, (n, M)),
            "producao_engorda": em_engorda, "vendida": vendida, "preco": preco,
            "receita_produtos": receita_produtos, "receita_servicos": receita_servicos,
            "receita": receita, "impostos_fat": impostos_fat, "receita_liquida": receita_liquida,
            "custo_racao": custo_racao, "custo_alevinos": custo_alevinos, "custo_energia": custo_energia,
            "outros_var": outros_var, "custos_variaveis": custos_variaveis, "lucro_bruto": lucro_bruto,
            "custos_fixos": custos_fixos, "ebitda": ebitda, "depreciacao": depreciacao, "lair": lair,
            "impostos_lucro": impostos_lucro, "lucro_liquido": lucro_liquido, "fco": fco,
        },
    }


def dre_mensal(eco: Dict[str, Any], i: int = 0) -> pd.DataFrame:
    s = eco["_series"]
    df = pd.DataFrame({
        "Mês": s["meses"], "Ano": s["ano"], "Capacidade (%)": s["capacidade"][i] * 100,
        "Produção em engorda (kg)": s["producao_engorda"][i], "Venda (kg)": s["vendida"][i],
        "Preço (R$/kg)": s["preco"][i], "Receita Produtos": s["receita_produtos"][i],
        "Receita Serviços": s["receita_servicos"][i], "Receita Bruta": s["receita"][i],
        "Impostos s/ Faturamento": s["impostos_fat"][i], "Receita Líquida": s["receita_liquida"][i],
        "Custo Ração": s["custo_racao"][i], "Custo Alevinos": s["custo_alevinos"][i],
        "Custo Energia": s["custo_energia"][i], "Outros Custos Variáveis": s["outros_var"][i],
        "Custos Variáveis": s["custos_variaveis"][i], "Lucro Bruto": s["lucro_bruto"][i],
        "Custos Fixos": s["custos_fixos"][i], "EBITDA": s["ebitda"][i], "Depreciação": s["depreciacao"][i],
        "LAIR": s["lair"][i], "Impostos s/ Lucro": s["impostos_lucro"][i],
        "Resultado Líquido": s["lucro_liquido"][i], "Fluxo de Caixa Operacional": s["fco"][i],
    })
    return df


def dre_anual(df_mensal: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in df_mensal.columns if c not in ("Mês", "Ano", "Capacidade (%)", "Preço (R$/kg)")]
    return df_mensal.groupby("Ano")[cols].sum().reset_index()


def distribuicao_sobras(cfg: EVTEASConfig, eco: Dict[str, Any]) -> Dict[str, float]:
    """Renda por cooperado (retirada + sobras) em regime — organizações cooperativas."""
    e = cfg.economico
    sobras = float(max(eco["lucro_liquido_regime"][0], 0.0))
    trab = sobras * e.perc_sobras_trabalhadores / 100
    forn = sobras - trab
    n_trab = max(e.cooperados_trabalhadores, 0)
    retirada_anual = e.mao_obra_mes * 12
    renda_mensal = safe_div(retirada_anual + trab, n_trab * 12, 0.0) if n_trab else 0.0
    return {
        "termo_resultado": "Lucro líquido" if e.tipo_organizacao == "empresa" else "Sobras líquidas",
        "sobras_regime": sobras,
        "sobras_trabalhadores": trab,
        "sobras_fornecedores": forn,
        "sobra_media_trabalhador": safe_div(trab, n_trab, 0.0),
        "sobra_media_fornecedor": safe_div(forn, e.cooperados_fornecedores, 0.0),
        "renda_mensal_por_trabalhador": float(renda_mensal),
    }


# ---------------------------------------------------------------------------
# 3. Dimensão ambiental
# ---------------------------------------------------------------------------

def fator_emissao_energia(cfg: EVTEASConfig) -> float:
    a = cfg.ambiental
    if a.fonte_energia == "mista":
        return FATORES_EMISSAO_ENERGIA["rede"] * (1 - a.fracao_renovavel_mista)
    if a.fonte_energia not in FATORES_EMISSAO_ENERGIA:
        raise ValueError(f"Fonte de energia desconhecida: {a.fonte_energia}")
    return FATORES_EMISSAO_ENERGIA[a.fonte_energia]


def score_conformidade(cfg: EVTEASConfig) -> Dict[str, Any]:
    a = cfg.ambiental
    itens = dict(a.checklist)
    itens["respeita_app"] = bool(itens.get("respeita_app", False)) and a.distancia_app_m >= a.faixa_app_minima_m
    total = sum(p for _, p, _ in CHECKLIST_AMBIENTAL.values())
    obtido = sum(p for k, (_, p, _) in CHECKLIST_AMBIENTAL.items() if itens.get(k, False))
    vetos = [CHECKLIST_AMBIENTAL[k][0] for k, (_, _, elim) in CHECKLIST_AMBIENTAL.items() if elim and not itens.get(k, False)]
    detalhe = [{"Item": d, "Peso": p, "Eliminatório": elim, "Atendido": bool(itens.get(k, False))}
               for k, (d, p, elim) in CHECKLIST_AMBIENTAL.items()]
    return {"score_0_10": 10 * obtido / total, "vetos": vetos, "detalhe": detalhe}


def calcular_ambiental(ctx: Contexto, tec: Dict[str, Any], eco: Dict[str, Any]) -> Dict[str, Any]:
    cfg, a, t = ctx.cfg, ctx.cfg.ambiental, ctx.cfg.tecnico
    prod = tec["producao_kg_ano"]
    prod_t = prod / 1000
    vol = tec["volume_util_m3"]
    disp = float(tec["disponibilidade"][0])

    # Gestão hídrica (m³/ano)
    enchimento = vol * disp * t.ciclos_ano if a.esvaziamento_por_ciclo else vol * disp / max(eco_horizonte(cfg), 1)
    renovacao = vol * disp * renovacao_pct_dia(cfg) / 100 * 365
    evaporacao = a.evaporacao_mm_dia / 1000 * t.area_lamina_m2 * disp * 365
    infiltracao = a.infiltracao_mm_dia / 1000 * t.area_lamina_m2 * disp * 365
    captada = enchimento + renovacao + evaporacao + infiltracao
    azul = evaporacao + a.fracao_retorno_perdida * (enchimento + renovacao)
    ph_azul = safe_div(azul, prod_t, np.nan)

    # Balanço de nutrientes e pegada hídrica cinza (poluente crítico)
    racao = tec["racao_ano_kg"]
    ganho = tec["ganho_biomassa_ano_kg"]
    n_in = racao * a.proteina_racao_pct / 100 / 6.25
    p_in = racao * a.fosforo_racao_pct / 100
    n_ret = ganho * a.nitrogenio_peixe_pct / 100
    p_ret = ganho * a.fosforo_peixe_pct / 100
    n_carga = np.maximum(n_in - n_ret, 0) * (1 - a.remocao_n_tratamento_pct / 100)
    p_carga = np.maximum(p_in - p_ret, 0) * (1 - a.remocao_p_tratamento_pct / 100)
    dn = a.n_max_mg_l - a.n_natural_mg_l
    dp = a.p_max_mg_l - a.p_natural_mg_l
    if dn <= 0 or dp <= 0:
        raise ValueError("Concentração máxima deve superar a natural (pegada hídrica cinza indefinida)")
    cinza_n = n_carga * 1000 / dn       # kg -> g; mg/L = g/m³
    cinza_p = p_carga * 1000 / dp
    cinza = np.maximum(cinza_n, cinza_p)
    critico = np.where(cinza_p >= cinza_n, "fósforo total", "nitrogênio total")

    # Energia e emissões (proxy de pegada de carbono)
    energia = energia_kwh_kg(cfg) * prod
    fe = fator_emissao_energia(cfg)
    co2_energia = energia * fe
    co2_racao = racao * a.fator_emissao_racao_kgco2_kg
    co2_diesel = np.full(ctx.n, a.diesel_l_ano * a.fator_emissao_diesel_kgco2_l)
    co2_total = co2_energia + co2_racao + co2_diesel
    intensidade = safe_div(co2_total, prod, np.nan)

    residuos = prod * a.residuos_kg_por_kg
    conf = score_conformidade(cfg)

    va = eco["valor_adicionado_regime"]
    # Ecoeficiência (WBCSD): valor adicionado / influência ambiental. Indefinida se VA <= 0.
    eco_carbono = np.where(va > 0, safe_div(va, co2_total, np.nan), np.nan)
    eco_agua = np.where(va > 0, safe_div(va, azul, np.nan), np.nan)

    return {
        "producao_t_ano": prod_t,
        "agua_captada_m3_ano": captada,
        "agua_enchimento_m3_ano": np.full(ctx.n, enchimento),
        "agua_renovacao_m3_ano": np.full(ctx.n, renovacao),
        "agua_evaporacao_m3_ano": np.full(ctx.n, evaporacao),
        "agua_infiltracao_m3_ano": np.full(ctx.n, infiltracao),
        "ph_azul_m3_ano": np.full(ctx.n, azul) if np.ndim(azul) == 0 else azul,
        "ph_azul_m3_t": ph_azul,
        "indice_uso_agua_kg_m3": safe_div(prod, captada, 0.0),
        "n_aportado_kg": n_in, "n_retido_kg": n_ret, "n_lancado_kg": n_carga,
        "p_aportado_kg": p_in, "p_retido_kg": p_ret, "p_lancado_kg": p_carga,
        "ph_cinza_m3_ano": cinza,
        "ph_cinza_m3_t": safe_div(cinza, prod_t, np.nan),
        "poluente_critico": critico,
        "energia_kwh_ano": energia,
        "fator_emissao_energia": fe,
        "co2e_energia_t": co2_energia / 1000,
        "co2e_racao_t": co2_racao / 1000,
        "co2e_diesel_t": co2_diesel / 1000,
        "co2e_total_t_ano": co2_total / 1000,
        "intensidade_carbono_kgco2e_kg": intensidade,
        "residuos_kg_ano": residuos,
        "residuos_reaproveitados_kg_ano": residuos * a.residuos_reaproveitados_pct / 100,
        "valor_adicionado_regime": va,
        "ecoeficiencia_r_por_kgco2e": eco_carbono,
        "ecoeficiencia_r_por_m3_azul": eco_agua,
        "ecoeficiencia_status": np.where(va > 0, "definida", "não definida (valor adicionado ≤ 0)"),
        "conformidade_0_10": conf["score_0_10"],
        "conformidade_detalhe": conf["detalhe"],
        "vetos_ambientais": conf["vetos"],
    }


def eco_horizonte(cfg: EVTEASConfig) -> int:
    return int(cfg.economico.horizonte_anos)


# ---------------------------------------------------------------------------
# 4. Dimensão social e governança
# ---------------------------------------------------------------------------

def calcular_social(ctx: Contexto, eco: Dict[str, Any]) -> Dict[str, Any]:
    cfg, s = ctx.cfg, ctx.cfg.social
    massa_anual = cfg.economico.mao_obra_mes * 12
    massa_local = massa_anual * s.mao_obra_local_pct / 100
    compras = (eco["custos_variaveis_regime"] + eco["custos_fixos_regime"] - eco["pessoal_regime"])
    compras_locais = np.maximum(compras, 0) * s.compras_locais_pct / 100
    custo_operacional = eco["custos_variaveis_regime"] + eco["custos_fixos_regime"]
    rvl = 100 * safe_div(massa_local + compras_locais, custo_operacional, 0.0)

    sobras = np.maximum(eco["lucro_liquido_regime"], 0) * cfg.economico.perc_sobras_trabalhadores / 100
    if cfg.economico.tipo_organizacao == "empresa":
        sobras = np.zeros(ctx.n)
    renda_local = massa_local + sobras * s.mao_obra_local_pct / 100
    iil = safe_div(renda_local, eco["receita_regime"], 0.0)
    n_trab = max(cfg.economico.cooperados_trabalhadores if cfg.economico.tipo_organizacao != "empresa"
                 else s.empregos_diretos, 1)
    renda_mensal = (massa_anual + sobras) / n_trab / 12
    renda_sm = renda_mensal / s.salario_minimo

    relacao = RELACAO_COMUNIDADE.get(s.relacao_comunidade)
    if relacao is None:
        raise ValueError(f"relacao_comunidade deve ser uma de {list(RELACAO_COMUNIDADE)}")
    participacao = 1.0 if cfg.governanca.itens.get("assembleias_regulares") else 0.0
    componentes_lso = {
        "Relação com a comunidade": relacao,
        "Canais formais de comunicação": 1.0 if s.canais_formais_comunicacao else 0.0,
        "Frequência de engajamento": min(s.reunioes_comunidade_ano / 12, 1.0),
        "Ausência de conflitos": math.exp(-0.5 * max(s.conflitos_registrados_ano, 0)),
        "Participação na governança": participacao,
    }
    lso = 100 * float(np.mean(list(componentes_lso.values())))
    risco = 1 / (1 + math.exp(8 * (lso / 100 - 0.5)))   # heurística logística (a calibrar)

    radar = {
        "Emprego local": np.full(ctx.n, s.mao_obra_local_pct / 100),
        "Renda": clamp((renda_sm - 0.5) / 1.5),
        "Segurança (NR)": np.full(ctx.n, s.conformidade_nr_pct / 100),
        "Capacitação": np.full(ctx.n, min(s.capacitacao_horas_pessoa_ano / 40, 1.0)),
        "Relação com a comunidade": np.full(ctx.n, relacao),
        "Inclusão": np.full(ctx.n, 0.5 * min(s.mulheres_pct / 50, 1) + 0.5 * min(s.jovens_pct / 30, 1)),
    }
    radar_score = np.mean(np.vstack(list(radar.values())), axis=0)

    return {
        "empregos_diretos": s.empregos_diretos,
        "empregos_indiretos": s.empregos_indiretos,
        "empregos_totais": s.empregos_diretos + s.empregos_indiretos,
        "massa_salarial_anual": massa_anual,
        "massa_salarial_local": massa_local,
        "compras_locais_regime": compras_locais,
        "rvl_pct": rvl,
        "indice_impacto_local": iil,
        "renda_mensal_trabalhador": renda_mensal,
        "renda_em_salarios_minimos": renda_sm,
        "lso_0_100": np.full(ctx.n, lso),
        "lso_componentes": componentes_lso,
        "risco_social": np.full(ctx.n, risco),
        "radar_social": radar,
        "radar_social_score": radar_score,
    }


def calcular_governanca(cfg: EVTEASConfig) -> Dict[str, Any]:
    itens = cfg.governanca.itens
    return {"score_0_1": safe_div(sum(bool(v) for v in itens.values()), len(itens), 0.0), "itens": dict(itens)}


def alinhamento_ods(cfg: EVTEASConfig, tec, eco, amb, soc, gov, i: int = 0):
    """Identificação automática dos ODS atendidos (regras explícitas e auditáveis)."""
    a, s = cfg.ambiental, cfg.social
    regras = [
        ("ODS 1 — Erradicação da pobreza", s.publico_vulneravel and soc["renda_em_salarios_minimos"][i] >= 1.0,
         "renda média por trabalhador ≥ 1 salário mínimo em público vulnerável"),
        ("ODS 2 — Fome zero e agricultura sustentável", tec["producao_kg_ano"][i] > 0,
         "produção de proteína animal de base aquícola"),
        ("ODS 5 — Igualdade de gênero", s.mulheres_pct >= 30, "participação feminina ≥ 30%"),
        ("ODS 6 — Água potável e saneamento", bool(a.checklist.get("tratamento_efluentes")) and bool(a.checklist.get("outorga_agua")),
         "tratamento de efluentes e outorga de uso da água"),
        ("ODS 7 — Energia limpa", a.fonte_energia in ("solar", "biomassa") or (a.fonte_energia == "mista" and a.fracao_renovavel_mista >= 0.5),
         "fonte de energia predominantemente renovável"),
        ("ODS 8 — Trabalho decente e crescimento econômico", s.conformidade_nr_pct >= 80 and soc["renda_em_salarios_minimos"][i] >= 1.0,
         "conformidade com NR ≥ 80% e renda ≥ 1 salário mínimo"),
        ("ODS 12 — Consumo e produção responsáveis", tec["fcr"][i] <= cfg.tecnico.fcr_referencia * 1.15 and a.residuos_reaproveitados_pct >= 50,
         "FCR até 15% acima da referência e reaproveitamento de resíduos ≥ 50%"),
        ("ODS 13 — Ação contra a mudança do clima", amb["intensidade_carbono_kgco2e_kg"][i] <= 2.0,
         "intensidade de carbono ≤ 2,0 kgCO2e/kg"),
        ("ODS 14 — Vida na água (meta 14.b: pescadores artesanais)", s.publico_vulneravel,
         "acesso de pescadores artesanais a recursos e mercados"),
        ("ODS 15 — Vida terrestre", a.distancia_app_m >= a.faixa_app_minima_m and bool(a.checklist.get("prevencao_escape")),
         "APP preservada e prevenção de escape de espécie exótica"),
        ("ODS 16 — Instituições eficazes", gov["score_0_1"] >= 0.7, "governança cooperativa ≥ 70% do checklist"),
    ]
    return pd.DataFrame([{"ODS": o, "Atendido": bool(c), "Critério": crit} for o, c, crit in regras])


# ---------------------------------------------------------------------------
# 5. Lean-Green: custo dos desperdícios evitáveis
# ---------------------------------------------------------------------------

def calcular_lean_green(ctx: Contexto, tec, eco) -> Dict[str, Any]:
    cfg, t = ctx.cfg, ctx.cfg.tecnico
    custo_racao = ctx.v("economico.custo_racao_kg")
    racao_exc = np.maximum(tec["fcr"] - t.fcr_referencia, 0) * tec["ganho_biomassa_ano_kg"]
    mort = ctx.v("tecnico.mortalidade_pct")
    peixes_perdidos = np.maximum(mort - t.mortalidade_referencia_pct, 0) / 100 * tec["alevinos_ano"]
    # perdas ocorrem, em média, na metade do ciclo (metade do peso final e da ração)
    custo_peixe = (ctx.v("economico.custo_alevino_milheiro") / 1000
                   + 0.5 * t.peso_final_g / 1000 * tec["fcr"] * custo_racao)
    kwh = energia_kwh_kg(cfg)
    energia_exc = max(kwh - cfg.ambiental.energia_referencia_kwh_kg, 0) * tec["producao_kg_ano"]
    custos = {
        "Ração excedente (superalimentação/FCR)": racao_exc * custo_racao,
        "Mortalidade evitável": peixes_perdidos * custo_peixe,
        "Energia excedente": energia_exc * ctx.v("economico.tarifa_energia_kwh"),
    }
    total = sum(custos.values())
    opex = eco["custos_variaveis_regime"] + eco["custos_fixos_regime"]
    return {"desperdicios_r_ano": custos, "total_r_ano": total, "pct_opex": safe_div(total, opex, 0.0),
            "racao_excedente_kg": racao_exc, "peixes_perdidos_evitaveis": peixes_perdidos,
            "energia_excedente_kwh": energia_exc}

## 4. Normalização, pesos (preset, Likert, AHP), índice EVTEAS, decisão e TOPSIS

In [ ]:
"""Normalização dos KPIs, ponderação (preset, Likert, AHP), índice EVTEAS,
regras de decisão e TOPSIS para comparação de alternativas."""

from typing import Any, Dict, List, Optional, Sequence

import numpy as np
import pandas as pd


DIMENSOES = ["tecnico", "economico", "ambiental", "social"]

# Referências de normalização explícitas (auditáveis na V&V). sentido: +1
# benefício (maior é melhor), -1 custo (menor é melhor).
REFERENCIAS_NORMALIZACAO = {
    "oee": (+1, 0.0, 1.0, "fração (Nakajima, 1988)"),
    "fcr": (-1, 1.0, 2.5, "kg ração/kg ganho"),
    "produtividade": (+1, 0.0, 1.0, "fração do potencial do sistema"),
    "vpl": (+1, -1.0, 1.0, "VPL / investimento total"),
    "roi": (+1, -0.10, 0.30, "lucro líquido anual / investimento"),
    "margem_seguranca": (+1, 0.0, 0.50, "fração da receita"),
    "ph_azul": (-1, 0.0, 3000.0, "m³/t"),
    "ph_cinza": (-1, 0.0, 100000.0, "m³/t"),
    "ecoeficiencia": (+1, 0.0, 10.0, "R$ de valor adicionado/kgCO2e"),
    "intensidade_carbono": (-1, 0.0, 4.0, "kgCO2e/kg"),
    "conformidade": (+1, 0.0, 10.0, "score 0–10"),
    "lso": (+1, 0.0, 100.0, "score 0–100 (Hurst; Ihlen, 2018)"),
    "rvl": (+1, 0.0, 100.0, "% do custo operacional (GRI 201-1/204-1)"),
    "radar_social": (+1, 0.0, 1.0, "média dos eixos do Radar Social"),
}


def normalizar(valor, sentido: int, minimo: float, maximo: float):
    v = np.nan_to_num(np.asarray(valor, dtype=float), nan=(minimo if sentido > 0 else maximo))
    if maximo <= minimo:
        return np.full_like(v, 0.5)
    z = (v - minimo) / (maximo - minimo) if sentido > 0 else (maximo - v) / (maximo - minimo)
    return np.clip(z, 0, 1)


def kpis_brutos(tec, eco, amb, soc) -> Dict[str, np.ndarray]:
    return {
        "oee": tec["oee"],
        "fcr": tec["fcr"],
        "produtividade": tec["produtividade_relativa"],
        "vpl": safe_div(eco["vpl"], eco["investimento_total"], 0.0),
        "roi": eco["roi_anual_regime"],
        "margem_seguranca": eco["margem_seguranca"],
        "ph_azul": amb["ph_azul_m3_t"],
        "ph_cinza": amb["ph_cinza_m3_t"],
        "ecoeficiencia": amb["ecoeficiencia_r_por_kgco2e"],   # NaN (indefinida) -> 0
        "intensidade_carbono": amb["intensidade_carbono_kgco2e_kg"],
        "conformidade": np.full_like(tec["oee"], amb["conformidade_0_10"]),
        "lso": soc["lso_0_100"],
        "rvl": soc["rvl_pct"],
        "radar_social": soc["radar_social_score"],
    }


# ---------------------------------------------------------------------------
# Pesos: Likert (Quadro 7), AHP e preset
# ---------------------------------------------------------------------------

def pesos_likert(avaliacoes: Dict[str, Dict[str, int]]) -> Dict[str, Any]:
    """Pesos relativos a partir da escala Likert de 1 a 5 (Seção 3.4).

    Para cada avaliador, peso = escore / soma dos escores; o peso final é a
    média entre avaliadores (triangulação). Informa a divergência por critério.
    """
    if not avaliacoes:
        raise ValueError("Nenhuma avaliação Likert informada")
    df = pd.DataFrame(avaliacoes).astype(float)          # linhas: critérios; colunas: avaliadores
    if df.isna().any().any():
        raise ValueError("Todos os avaliadores devem pontuar todos os critérios")
    if ((df < 1) | (df > 5)).any().any() or (df % 1 != 0).any().any():
        raise ValueError("Escores Likert devem ser inteiros entre 1 e 5")
    pesos_av = df / df.sum(axis=0)
    pesos = pesos_av.mean(axis=1)
    pesos = pesos / pesos.sum()
    tabela = df.copy()
    tabela.columns = [f"Escore — {c}" for c in df.columns]
    tabela["Divergência (máx − mín)"] = df.max(axis=1) - df.min(axis=1)
    tabela["Peso normalizado"] = pesos
    return {"pesos": pesos.to_dict(), "tabela": tabela.reset_index().rename(columns={"index": "Critério"}),
            "concordancia_media": float(1 - (df.max(axis=1) - df.min(axis=1)).mean() / 4)}


INDICE_ALEATORIO = {1: 0.0, 2: 0.0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49}


def pesos_ahp(matriz: Sequence[Sequence[float]]) -> Dict[str, Any]:
    """Pesos AHP (autovetor principal) e razão de consistência (Saaty)."""
    A = np.asarray(matriz, dtype=float)
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("Matriz AHP deve ser quadrada")
    if np.any(A <= 0) or not np.allclose(A * A.T, 1.0, atol=1e-6):
        raise ValueError("Matriz AHP deve ser positiva e recíproca (a_ji = 1/a_ij)")
    vals, vecs = np.linalg.eig(A)
    k = int(np.argmax(vals.real))
    lam = float(vals.real[k])
    w = np.abs(vecs[:, k].real)
    w = w / w.sum()
    n = A.shape[0]
    ci = (lam - n) / (n - 1) if n > 1 else 0.0
    ri = INDICE_ALEATORIO.get(n, 1.49)
    cr = ci / ri if ri > 0 else 0.0
    return {"pesos": w, "lambda_max": lam, "ic": ci, "rc": cr, "consistente": cr <= 0.10}


def resolver_pesos(cfg: EVTEASConfig) -> Dict[str, Any]:
    p = cfg.pesos
    info: Dict[str, Any] = {"metodo": p.metodo}
    if p.metodo == "preset":
        dim = p.vetor()
    elif p.metodo == "likert":
        lk = pesos_likert(p.likert_dimensoes)
        faltando = set(DIMENSOES) - set(lk["pesos"])
        if faltando:
            raise ValueError(f"Likert sem as dimensões: {faltando}")
        dim = {d: lk["pesos"][d] for d in DIMENSOES}
        info["likert_dimensoes"] = lk
    elif p.metodo == "ahp":
        if p.ahp_matriz is None:
            raise ValueError("Método AHP exige ahp_matriz (ordem: técnico, econômico, ambiental, social)")
        ahp = pesos_ahp(p.ahp_matriz)
        dim = dict(zip(DIMENSOES, ahp["pesos"].tolist()))
        info["ahp"] = ahp
    else:
        raise ValueError(f"Método de ponderação desconhecido: {p.metodo}")

    kpi: Dict[str, float] = {}
    if p.likert_kpis:
        lk = pesos_likert(p.likert_kpis)
        info["likert_kpis"] = lk
        kpi_bruto = lk["pesos"]
    else:
        kpi_bruto = p.kpi
    for d, nomes in KPIS_POR_DIMENSAO.items():
        w = np.array([max(float(kpi_bruto.get(k, 1.0)), 0.0) for k in nomes])
        w = w / w.sum() if w.sum() > 0 else np.full(len(nomes), 1 / len(nomes))
        kpi.update(dict(zip(nomes, w.tolist())))
    info["dimensoes"] = dim
    info["kpi_intra_dimensao"] = kpi
    return info


# ---------------------------------------------------------------------------
# Índice integrado, ESG e decisão
# ---------------------------------------------------------------------------

def calcular_indice(cfg: EVTEASConfig, tec, eco, amb, soc, gov, pesos=None) -> Dict[str, Any]:
    pesos = pesos or resolver_pesos(cfg)
    brutos = kpis_brutos(tec, eco, amb, soc)
    norm = {k: normalizar(v, *REFERENCIAS_NORMALIZACAO[k][:3]) for k, v in brutos.items()}
    dims = {}
    for d, nomes in KPIS_POR_DIMENSAO.items():
        dims[d] = sum(norm[k] * pesos["kpi_intra_dimensao"][k] for k in nomes)
    indice = sum(dims[d] * pesos["dimensoes"][d] for d in DIMENSOES)
    esg = {"E": dims["ambiental"], "S": dims["social"], "G": np.full_like(indice, gov["score_0_1"])}
    esg_score = (esg["E"] + esg["S"] + esg["G"]) / 3
    return {"kpis_brutos": brutos, "kpis_normalizados": norm, "dimensoes": dims,
            "indice_evteas": np.clip(indice, 0, 1), "esg": esg, "esg_score": esg_score, "pesos": pesos}


def classificar(cfg: EVTEASConfig, vpl: float, indice: float, lso: float, vetos_ambientais: List[str],
                prob_vpl_positivo: Optional[float] = None) -> Dict[str, Any]:
    """Regra de decisão em dois estágios: vetos não compensatórios + índice compensatório.

    O índice agregado sozinho pode ocultar fragilidades (Rev. 186, Seção 5.4);
    por isso VPL negativo, item ambiental eliminatório não atendido ou LSO abaixo
    do mínimo impedem a classificação "VIÁVEL", qualquer que seja o índice.
    """
    d = cfg.decisao
    motivos = []
    if d.aplicar_vetos:
        if vpl < 0:
            motivos.append("VPL negativo (veto econômico)")
        motivos += [f"Item eliminatório não atendido: {v}" for v in vetos_ambientais]
        if lso < d.lso_minimo:
            motivos.append(f"LSO abaixo do mínimo ({lso:.0f} < {d.lso_minimo:.0f})")
    ressalvas = []
    if prob_vpl_positivo is not None and prob_vpl_positivo < d.prob_vpl_positivo_minima:
        ressalvas.append(f"P(VPL > 0) = {prob_vpl_positivo:.0%} abaixo de {d.prob_vpl_positivo_minima:.0%}")
    if motivos:
        classe = "NÃO VIÁVEL"
    elif indice >= d.limiar_indice_viavel and not ressalvas:
        classe = "VIÁVEL"
    elif indice >= d.limiar_indice_ressalvas:
        classe = "VIÁVEL COM RESSALVAS"
        if indice < d.limiar_indice_viavel:
            ressalvas.append(f"Índice EVTEAS {indice:.3f} abaixo do limiar {d.limiar_indice_viavel:.2f}")
    else:
        classe = "NÃO VIÁVEL"
        motivos.append(f"Índice EVTEAS {indice:.3f} abaixo de {d.limiar_indice_ressalvas:.2f}")
    return {"classificacao": classe, "vetos": motivos, "ressalvas": ressalvas}


# ---------------------------------------------------------------------------
# TOPSIS — comparação de alternativas de projeto
# ---------------------------------------------------------------------------

def topsis(df: pd.DataFrame, criterios: List[str], pesos: Sequence[float], beneficio: Sequence[bool]) -> pd.DataFrame:
    X = df[criterios].astype(float).to_numpy()
    if np.isnan(X).any():
        raise ValueError("TOPSIS não aceita valores ausentes na matriz de decisão")
    W = np.asarray(pesos, dtype=float)
    W = W / W.sum()
    den = np.sqrt((X ** 2).sum(axis=0))
    den[den == 0] = 1
    V = X / den * W
    ben = np.asarray(beneficio, dtype=bool)
    ideal = np.where(ben, V.max(axis=0), V.min(axis=0))
    anti = np.where(ben, V.min(axis=0), V.max(axis=0))
    d_mais = np.sqrt(((V - ideal) ** 2).sum(axis=1))
    d_menos = np.sqrt(((V - anti) ** 2).sum(axis=1))
    out = df.copy()
    out["D+"] = d_mais
    out["D-"] = d_menos
    out["TOPSIS"] = d_menos / (d_mais + d_menos + 1e-15)
    out["Ranking"] = out["TOPSIS"].rank(ascending=False, method="min").astype(int)
    return out.sort_values("Ranking").reset_index(drop=True)

## 5. Incerteza: Monte Carlo, sensibilidade, valores críticos e cenários

In [ ]:
"""Tratamento da incerteza: Monte Carlo, sensibilidade, valores críticos e cenários."""

import copy
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd



def distribuicoes_efetivas(cfg: EVTEASConfig) -> Dict[str, Distribuicao]:
    d = dict(cfg.monte_carlo.distribuicoes)
    if cfg.monte_carlo.incluir_capex_aace and "economico.capex_fator" not in d:
        lo, hi = FAIXAS_AACE.get(int(cfg.economico.classe_estimativa_aace), FAIXAS_AACE[5])
        d["economico.capex_fator"] = Distribuicao("triangular", 1 + lo, 1.0, 1 + hi)
    for caminho, dist in d.items():
        obter_por_caminho(cfg, caminho)  # valida o caminho
        dist.validar()
    return d


def amostrar(cfg: EVTEASConfig, n: int, seed: int) -> Dict[str, np.ndarray]:
    rng = np.random.default_rng(seed)
    # ordem fixa (alfabética) => reprodutibilidade independente da ordem de inserção
    return {k: dist.amostrar(rng, n) for k, dist in sorted(distribuicoes_efetivas(cfg).items())}


def monte_carlo(cfg: EVTEASConfig, n: Optional[int] = None, seed: Optional[int] = None) -> Dict[str, Any]:

    n = int(n or cfg.monte_carlo.iteracoes)
    seed = int(cfg.monte_carlo.seed if seed is None else seed)
    entradas = amostrar(cfg, n, seed)
    r = motor(cfg, entradas, n)
    eco, tec, amb, idx = r["economico"], r["tecnico"], r["ambiental"], r["indice"]
    tma = cfg.economico.tma_aa_pct / 100

    amostra = pd.DataFrame({k: v for k, v in entradas.items()})
    amostra["VPL"] = eco["vpl"]
    amostra["TIR_aa"] = eco["tir_anual"]
    amostra["Diagnostico_TIR"] = eco["diagnostico_tir"]
    amostra["Payback_desc_meses"] = eco["payback_descontado_meses"]
    amostra["IL"] = eco["indice_lucratividade"]
    amostra["VPL_beneficiario"] = eco["vpl_beneficiario"]
    amostra["ROI_anual"] = eco["roi_anual_regime"]
    amostra["Producao_kg_ano"] = tec["producao_kg_ano"]
    amostra["OEE"] = tec["oee"]
    amostra["PH_cinza_m3_t"] = amb["ph_cinza_m3_t"]
    amostra["Intensidade_kgCO2e_kg"] = amb["intensidade_carbono_kgco2e_kg"]
    amostra["Indice_EVTEAS"] = idx["indice_evteas"]

    v = amostra["VPL"].to_numpy()
    p5 = np.percentile(v, 5)
    media, dp = float(v.mean()), float(v.std(ddof=1))
    erro = dp / np.sqrt(n)
    tir_valida = amostra["TIR_aa"].to_numpy()
    tir_valida = tir_valida[np.isfinite(tir_valida)]
    estat = {
        "n": n, "seed": seed,
        "media_vpl": media, "mediana_vpl": float(np.median(v)), "desvio_vpl": dp,
        "variancia_vpl": dp ** 2, "cv_vpl": dp / abs(media) if media else float("nan"),
        "ic95_media_vpl": (media - 1.96 * erro, media + 1.96 * erro),
        "prob_vpl_negativo": float((v < 0).mean()),
        "prob_vpl_positivo": float((v > 0).mean()),
        "prob_tir_maior_tma": float((amostra["TIR_aa"] > tma).mean()),
        "media_vpl_beneficiario": float(amostra["VPL_beneficiario"].mean()),
        "prob_vpl_beneficiario_positivo": float((amostra["VPL_beneficiario"] > 0).mean()),
        "fracao_tir_indefinida": float((~np.isfinite(amostra["TIR_aa"])).mean()),
        "var5_vpl": float(p5),
        "cvar5_vpl": float(v[v <= p5].mean()),
        "percentis_vpl": percentis(v),
        "percentis_tir": percentis(tir_valida),
        "percentis_payback_desc": percentis(amostra["Payback_desc_meses"]),
        "percentis_indice": percentis(amostra["Indice_EVTEAS"]),
        "media_indice": float(amostra["Indice_EVTEAS"].mean()),
        "prob_indice_maior_limiar": float((amostra["Indice_EVTEAS"] >= cfg.decisao.limiar_indice_viavel).mean()),
    }
    # Convergência: média acumulada e erro-padrão em pontos de controle
    pontos = sorted({int(x) for x in np.unique(np.geomspace(100, n, num=12).astype(int)) if x <= n} | {n})
    conv = pd.DataFrame([{"Iterações": k, "Média VPL": float(v[:k].mean()),
                          "Erro-padrão": float(v[:k].std(ddof=1) / np.sqrt(k)),
                          "P(VPL>0)": float((v[:k] > 0).mean())} for k in pontos])
    # Importância: correlação de postos (Spearman) entre entradas e saídas
    ranks = amostra[list(entradas) + ["VPL", "Indice_EVTEAS"]].rank()
    constantes = [k for k in ranks if ranks[k].nunique() <= 1]
    ranks[constantes] = np.nan       # correlação indefinida para entradas sem variação
    imp = pd.DataFrame({
        "Variável": list(entradas),
        "Spearman com VPL": [float(ranks[k].corr(ranks["VPL"])) for k in entradas],
        "Spearman com Índice": [float(ranks[k].corr(ranks["Indice_EVTEAS"])) for k in entradas],
    })
    imp = imp.reindex(imp["Spearman com VPL"].abs().sort_values(ascending=False).index).reset_index(drop=True)
    return {"amostra": amostra, "estatisticas": estat, "convergencia": conv, "importancia": imp,
            "distribuicoes": {k: vars(d) for k, d in distribuicoes_efetivas(cfg).items()}}


# ---------------------------------------------------------------------------
# Sensibilidade determinística (um fator por vez)
# ---------------------------------------------------------------------------

VARIAVEIS_SENSIBILIDADE = [
    "economico.preco_venda_kg", "tecnico.fcr", "economico.custo_racao_kg", "tecnico.mortalidade_pct",
    "tecnico.desempenho_crescimento_pct", "economico.capex_fator", "economico.custos_fixos_fator",
    "economico.tarifa_energia_kwh", "economico.custo_alevino_milheiro", "economico.tma_aa_pct",
]


def _vpl_com(cfg: EVTEASConfig, caminho: str, valor: float) -> float:
    if caminho == "economico.tma_aa_pct":
        c = copy.deepcopy(cfg)
        definir_por_caminho(c, caminho, valor)
        return float(motor(c, None, 1)["economico"]["vpl"][0])
    return float(motor(cfg, {caminho: np.array([valor])}, 1)["economico"]["vpl"][0])


def sensibilidade(cfg: EVTEASConfig, variaveis: Optional[List[str]] = None,
                  amplitudes=(0.10, 0.20)) -> pd.DataFrame:
    variaveis = variaveis or VARIAVEIS_SENSIBILIDADE
    base = float(motor(cfg, None, 1)["economico"]["vpl"][0])
    linhas = []
    for var in variaveis:
        x = float(obter_por_caminho(cfg, var))
        for amp in amplitudes:
            d = abs(x) * amp if x != 0 else amp
            lo, hi = _vpl_com(cfg, var, x - d), _vpl_com(cfg, var, x + d)
            elasticidade = ((hi - lo) / base) / (2 * d / x) if (base and x) else float("nan")
            linhas.append({"Variável": var, "Amplitude": amp, "Valor base": x, "VPL base": base,
                           "VPL (−)": lo, "VPL (+)": hi, "Δ VPL (−)": lo - base, "Δ VPL (+)": hi - base,
                           "Amplitude do efeito": abs(hi - lo),
                           "Δ VPL por 1% da variável": (hi - lo) / (2 * amp * 100),
                           "Elasticidade": elasticidade})
    df = pd.DataFrame(linhas)
    ordem = df[df["Amplitude"] == amplitudes[0]].sort_values("Amplitude do efeito", ascending=False)["Variável"]
    df["_o"] = df["Variável"].map({v: i for i, v in enumerate(ordem)})
    return df.sort_values(["_o", "Amplitude"]).drop(columns="_o").reset_index(drop=True)


def valores_criticos(cfg: EVTEASConfig, variaveis: Optional[List[str]] = None) -> pd.DataFrame:
    """Valor de cada variável que zera o VPL (demais premissas no caso-base)."""
    variaveis = variaveis or [v for v in VARIAVEIS_SENSIBILIDADE if v != "economico.tma_aa_pct"]
    linhas = []
    for var in variaveis:
        x = float(obter_por_caminho(cfg, var))
        a, b = 0.0, max(x * 5, 1.0)
        fa, fb = _vpl_com(cfg, var, a), _vpl_com(cfg, var, b)
        if var == "tecnico.mortalidade_pct":
            b = 99.0
            fb = _vpl_com(cfg, var, b)
        critico = float("nan")
        if fa * fb <= 0:
            for _ in range(80):
                m = (a + b) / 2
                fm = _vpl_com(cfg, var, m)
                if fa * fm <= 0:
                    b, fb = m, fm
                else:
                    a, fa = m, fm
            critico = (a + b) / 2
        folga = (critico - x) / x if (x and np.isfinite(critico)) else float("nan")
        linhas.append({"Variável": var, "Valor base": x, "Valor crítico (VPL = 0)": critico,
                       "Folga relativa": folga})
    return pd.DataFrame(linhas)


def cenarios(cfg: EVTEASConfig) -> pd.DataFrame:
    """Cenários pessimista, base e otimista a partir dos extremos das distribuições."""
    dists = distribuicoes_efetivas(cfg)
    base = motor(cfg, None, 1)["economico"]["vpl"][0]
    pess, otim = {}, {}
    for k, d in dists.items():
        vmin = _vpl_com(cfg, k, d.minimo) if k != "economico.tma_aa_pct" else base
        vmax = _vpl_com(cfg, k, d.maximo) if k != "economico.tma_aa_pct" else base
        favoravel_max = vmax >= vmin
        pess[k] = np.array([d.minimo if favoravel_max else d.maximo])
        otim[k] = np.array([d.maximo if favoravel_max else d.minimo])
    linhas = []
    for nome, ov in (("Pessimista", pess), ("Base", None), ("Otimista", otim)):
        r = motor(cfg, ov, 1)
        e, i = r["economico"], r["indice"]
        linhas.append({"Cenário": nome, "VPL": float(e["vpl"][0]), "TIR a.a.": float(e["tir_anual"][0]),
                       "Payback descontado (meses)": float(e["payback_descontado_meses"][0]),
                       "Produção (kg/ano)": float(r["tecnico"]["producao_kg_ano"][0]),
                       "Índice EVTEAS": float(i["indice_evteas"][0])})
    return pd.DataFrame(linhas)

## 6. Pipeline principal reproduzível

In [ ]:
"""Pipeline principal reproduzível do EVTEAS-Py."""

import copy
from dataclasses import asdict
from typing import Any, Dict, Optional

import numpy as np
import pandas as pd



def motor(cfg: EVTEASConfig, overrides: Optional[Dict[str, np.ndarray]] = None, n: int = 1,
          pesos: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Executa as quatro dimensões e o índice para n realizações das premissas."""
    ctx = Contexto(cfg, overrides, n)
    tec = calcular_tecnico(ctx)
    eco = calcular_economico(ctx, tec)
    amb = calcular_ambiental(ctx, tec, eco)
    soc = calcular_social(ctx, eco)
    gov = calcular_governanca(cfg)
    lean = calcular_lean_green(ctx, tec, eco)
    idx = calcular_indice(cfg, tec, eco, amb, soc, gov, pesos)
    return {"tecnico": tec, "economico": eco, "ambiental": amb, "social": soc,
            "governanca": gov, "lean_green": lean, "indice": idx}


def _escalar(obj, i: int = 0):
    """Extrai a realização i (determinística) de estruturas com vetores."""
    if isinstance(obj, dict):
        return {k: _escalar(v, i) for k, v in obj.items() if k != "_series"}
    if isinstance(obj, np.ndarray):
        if obj.ndim == 1:
            v = obj[i]
            return v.item() if hasattr(v, "item") else v
        return obj
    return obj


def tabela_premissas(cfg: EVTEASConfig) -> pd.DataFrame:
    """Registro de origem de cada premissa (Seção 3.6): valor, fonte e distribuição."""
    dists = distribuicoes_efetivas(cfg)
    linhas = []
    for caminho in caminhos_numericos(cfg) + [k for k in dists if k not in caminhos_numericos(cfg)]:
        fonte = cfg.fontes.get(caminho)
        d = dists.get(caminho)
        linhas.append({
            "Parâmetro": caminho, "Valor": obter_por_caminho(cfg, caminho),
            "Categoria da fonte": CATEGORIAS_FONTE[fonte.categoria] if fonte else CATEGORIAS_FONTE["autor"],
            "Referência": fonte.referencia if fonte else "",
            "Distribuição": d.tipo if d else "determinístico",
            "Mínimo": d.minimo if d else None, "Mais provável": d.moda if d else None, "Máximo": d.maximo if d else None,
        })
    return pd.DataFrame(linhas)


def executar_evteas(cfg: EVTEASConfig, executar_mc: bool = True, executar_sens: bool = True,
                    n_mc: Optional[int] = None) -> Dict[str, Any]:
    if not isinstance(cfg, EVTEASConfig):
        raise TypeError("cfg deve ser uma instância de EVTEASConfig")

    pesos = resolver_pesos(cfg)
    r = motor(cfg, None, 1, pesos)
    df_m = dre_mensal(r["economico"])
    det = {k: _escalar(v) for k, v in r.items()}
    resultado: Dict[str, Any] = {
        "versao": VERSAO,
        "config": asdict(cfg),
        **det,
        "curva_crescimento": curva_crescimento(cfg),
        "dre_mensal": df_m,
        "dre_anual": dre_anual(df_m),
        "fluxo_caixa": r["economico"]["fluxo_caixa"][0],
        "sobras": distribuicao_sobras(cfg, r["economico"]),
        "ods": alinhamento_ods(cfg, r["tecnico"], r["economico"], r["ambiental"], r["social"], r["governanca"]),
        "premissas": tabela_premissas(cfg),
    }
    prob = None
    if executar_mc:
        resultado["monte_carlo"] = monte_carlo(cfg, n=n_mc)
        prob = resultado["monte_carlo"]["estatisticas"]["prob_vpl_positivo"]
    if executar_sens:
        resultado["sensibilidade"] = sensibilidade(cfg)
        resultado["valores_criticos"] = valores_criticos(cfg)
        resultado["cenarios"] = cenarios(cfg)
    resultado["decisao"] = classificar(cfg, det["economico"]["vpl"], det["indice"]["indice_evteas"],
                                       det["social"]["lso_0_100"], det["ambiental"]["vetos_ambientais"], prob)
    resultado["vv"] = validar_invariantes(cfg, resultado, r)
    return resultado


def comparar_alternativas(alternativas: Dict[str, EVTEASConfig], pesos_criterios: Optional[Dict[str, float]] = None,
                          n_mc: int = 2000) -> Dict[str, Any]:
    """Executa cada alternativa e as ordena por TOPSIS (critérios das quatro dimensões)."""
    criterios = {  # nome: (extrator, benefício?)
        "VPL (R$)": (lambda r, m: r["economico"]["vpl"], True),
        "P(VPL>0)": (lambda r, m: m["estatisticas"]["prob_vpl_positivo"], True),
        "OEE": (lambda r, m: r["tecnico"]["oee"], True),
        "PH cinza (m³/t)": (lambda r, m: r["ambiental"]["ph_cinza_m3_t"], False),
        "Intensidade carbono (kgCO2e/kg)": (lambda r, m: r["ambiental"]["intensidade_carbono_kgco2e_kg"], False),
        "LSO": (lambda r, m: r["social"]["lso_0_100"], True),
        "RVL (%)": (lambda r, m: r["social"]["rvl_pct"], True),
    }
    pesos_criterios = pesos_criterios or {"VPL (R$)": 0.20, "P(VPL>0)": 0.15, "OEE": 0.15,
                                          "PH cinza (m³/t)": 0.15, "Intensidade carbono (kgCO2e/kg)": 0.10,
                                          "LSO": 0.10, "RVL (%)": 0.15}
    linhas, resultados = [], {}
    for nome, cfg in alternativas.items():
        r = executar_evteas(cfg, executar_mc=False, executar_sens=False)
        m = monte_carlo(cfg, n=n_mc)
        resultados[nome] = r
        linha = {"Alternativa": nome}
        for c, (f, _) in criterios.items():
            linha[c] = float(f(r, m))
        linha["Índice EVTEAS"] = r["indice"]["indice_evteas"]
        linha["Classificação"] = classificar(cfg, r["economico"]["vpl"], r["indice"]["indice_evteas"],
                                             r["social"]["lso_0_100"], r["ambiental"]["vetos_ambientais"],
                                             m["estatisticas"]["prob_vpl_positivo"])["classificacao"]
        linhas.append(linha)
    df = pd.DataFrame(linhas)
    nomes = list(criterios)
    ranking = topsis(df, nomes, [pesos_criterios[c] for c in nomes], [criterios[c][1] for c in nomes])
    return {"matriz": df, "ranking": ranking, "pesos": pesos_criterios, "resultados": resultados}


def copiar(cfg: EVTEASConfig) -> EVTEASConfig:
    return copy.deepcopy(cfg)

## 7. Verificação (V&V): invariantes e testes de regressão

In [ ]:
"""Verificação computacional (Boehm, 1984): invariantes e testes de regressão.

A verificação responde "estamos construindo o produto corretamente?". A
validação junto aos especialistas (Seção 3.3.2) é conduzida fora do código,
mas seus escores Likert entram em ``PesosConfig`` e ficam rastreáveis.
"""

from typing import Any, Dict, List

import numpy as np



def validar_invariantes(cfg: EVTEASConfig, resultado: Dict[str, Any], bruto: Dict[str, Any] = None) -> Dict[str, Any]:
    checks: List[Dict[str, Any]] = []

    def chk(nome: str, ok: bool, detalhe: str = ""):
        checks.append({"Verificação": nome, "Aprovada": bool(ok), "Detalhe": detalhe})

    t, e, a, s, i = (resultado["tecnico"], resultado["economico"], resultado["ambiental"],
                     resultado["social"], resultado["indice"])
    chk("OEE em [0, 1]", 0 <= t["oee"] <= 1, f"OEE = {t['oee']:.4f}")
    chk("OEE = Disponibilidade × Performance × Qualidade",
        abs(t["oee"] - t["disponibilidade"] * t["performance"] * t["qualidade"]) < 1e-12)
    chk("FCR não negativo", t["fcr"] >= 0)
    chk("Ciclos/ano compatíveis com a duração do ciclo",
        cfg.tecnico.ciclos_ano <= 365 / max(cfg.tecnico.ciclo_dias, 1) + 1e-9,
        f"{cfg.tecnico.ciclos_ano} ≤ {365 / cfg.tecnico.ciclo_dias:.2f}")
    chk("Biomassa de despesca ≤ capacidade de suporte",
        t["biomassa_despesca_ciclo_kg"] <= t["volume_util_m3"] * t["capacidade_suporte_kg_m3"] * t["disponibilidade"] + 1e-6)
    chk("Ração = FCR × ganho de biomassa", abs(t["racao_ciclo_kg"] - t["fcr"] * t["ganho_biomassa_ciclo_kg"]) < 1e-6)

    df = resultado["dre_mensal"]
    close = lambda x, y: bool(np.allclose(x, y, rtol=1e-9, atol=1e-6))
    chk("DRE: Receita Líquida = Receita Bruta − Impostos s/ Faturamento",
        close(df["Receita Líquida"], df["Receita Bruta"] - df["Impostos s/ Faturamento"]))
    chk("DRE: Lucro Bruto = Receita Líquida − Custos Variáveis",
        close(df["Lucro Bruto"], df["Receita Líquida"] - df["Custos Variáveis"]))
    chk("DRE: EBITDA = Lucro Bruto − Custos Fixos", close(df["EBITDA"], df["Lucro Bruto"] - df["Custos Fixos"]))
    chk("DRE: Resultado = LAIR − Impostos s/ Lucro", close(df["Resultado Líquido"], df["LAIR"] - df["Impostos s/ Lucro"]))
    chk("Fluxo de caixa operacional = Resultado + Depreciação",
        close(df["Fluxo de Caixa Operacional"], df["Resultado Líquido"] + df["Depreciação"]))
    chk("Conservação de massa: produção vendida ≤ produção em engorda",
        df["Venda (kg)"].sum() <= df["Produção em engorda (kg)"].sum() + 1e-6)

    fc = resultado["fluxo_caixa"]
    chk("VPL recalculado de forma independente", abs(vpl(e["tma_mensal"], fc) - e["vpl"]) < 1e-6)
    if np.isfinite(e["tir_mensal"]):
        chk("VPL na TIR ≈ 0", abs(vpl(e["tir_mensal"], fc)) < 1e-3 * max(1.0, abs(fc[0])), e["diagnostico_tir"])
    else:
        chk("TIR indefinida acompanhada de diagnóstico", e["diagnostico_tir"] == "sem_solucao",
            e["diagnostico_tir"])
    chk("Payback descontado ≥ payback simples (quando ambos existem)",
        not (np.isfinite(e["payback_descontado_meses"]) and np.isfinite(e["payback_simples_meses"]))
        or e["payback_descontado_meses"] >= e["payback_simples_meses"] - 1e-9)

    chk("Balanço de N: aportado ≥ retido", a["n_aportado_kg"] >= a["n_retido_kg"])
    chk("Balanço de P: aportado ≥ retido", a["p_aportado_kg"] >= a["p_retido_kg"])
    chk("Score de conformidade em [0, 10]", 0 <= a["conformidade_0_10"] <= 10)
    chk("Ecoeficiência indefinida somente com valor adicionado ≤ 0",
        (a["ecoeficiencia_status"] == "definida") == (a["valor_adicionado_regime"] > 0))
    chk("LSO em [0, 100]", 0 <= s["lso_0_100"] <= 100)
    chk("Risco social em [0, 1]", 0 <= s["risco_social"] <= 1)

    pesos = i["pesos"]
    chk("Pesos dimensionais somam 1", abs(sum(pesos["dimensoes"].values()) - 1) < 1e-9)
    chk("Pesos dimensionais não negativos", min(pesos["dimensoes"].values()) >= 0)
    chk("Índice EVTEAS em [0, 1]", 0 <= i["indice_evteas"] <= 1, f"{i['indice_evteas']:.4f}")
    chk("KPIs normalizados em [0, 1]", all(0 <= v <= 1 for v in i["kpis_normalizados"].values()))
    if "ahp" in pesos:
        chk("AHP consistente (RC ≤ 0,10)", pesos["ahp"]["consistente"], f"RC = {pesos['ahp']['rc']:.3f}")

    if "monte_carlo" in resultado:
        mc = resultado["monte_carlo"]
        st = mc["estatisticas"]
        chk("Monte Carlo: P(VPL>0) + P(VPL<0) ≤ 1", st["prob_vpl_positivo"] + st["prob_vpl_negativo"] <= 1 + 1e-12)
        chk("Monte Carlo: P5 ≤ mediana ≤ P95",
            st["percentis_vpl"]["p5"] <= st["mediana_vpl"] <= st["percentis_vpl"]["p95"])
        conv = mc["convergencia"]
        chk("Monte Carlo: erro-padrão final < 2% do desvio-padrão",
            conv["Erro-padrão"].iloc[-1] < 0.02 * st["desvio_vpl"] or st["n"] < 2500,
            f"EP = {conv['Erro-padrão'].iloc[-1]:,.0f}")

    falhas = [c for c in checks if not c["Aprovada"]]
    return {"valido": not falhas, "quantidade_verificacoes": len(checks), "quantidade_erros": len(falhas),
            "erros": [c["Verificação"] for c in falhas], "verificacoes": checks}


def teste_regressao_deterministica() -> bool:
    """Casos com solução analítica conhecida (executáveis sem pytest)."""
    assert abs(vpl(0.0, [-100, 50, 50]) - 0.0) < 1e-12
    assert abs(vpl(0.10, [-100, 110]) - 0.0) < 1e-9
    r, d = tir([-100, 110])
    assert abs(r[0] - 0.10) < 1e-9 and d[0] == "convencional"
    r, d = tir([-100, -10, -10])
    assert np.isnan(r[0]) and d[0] == "sem_solucao"
    assert abs(payback([-100, 40, 40, 40])[0] - 2.5) < 1e-12
    assert np.isnan(payback([-100, 10, 10])[0])
    return True

## 8. Caso-base, alternativas e preset legado da Rev. 186

In [ ]:
"""Configurações de referência: caso-base ilustrativo, alternativas e preset legado.

O caso-base representa um Projeto de Geração de Trabalho e Renda (PGTR) de
piscicultura de tilápia em viveiros escavados, conduzido por cooperativa de
pescadores artesanais. Os valores são estimativas ilustrativas, registradas
com sua origem; não constituem dados de um empreendimento real e devem ser
recalibrados na validação com o especialista em piscicultura.
"""

import copy


REF_KUBITZA = "Kubitza (2017) — faixa zootécnica de referência para tilápia"
REF_CONAMA = "Resolução CONAMA nº 357/2005, classe 2, ambiente lótico"
REF_COTACAO = "Estimativa ilustrativa; substituir por cotação regional atualizada"


def criar_config_caso_base() -> EVTEASConfig:
    cfg = EVTEASConfig(projeto="Caso-base — PGTR de piscicultura de tilápia em viveiros escavados")
    cfg.tecnico = TecnicoConfig()
    cfg.economico = EconomicoConfig()
    cfg.ambiental = AmbientalConfig()
    cfg.social = SocialConfig()
    cfg.economico.fomento_nao_reembolsavel_pct = 70.0
    cfg.fontes = {
        "economico.fomento_nao_reembolsavel_pct": Fonte("autor", "Hipótese de aporte por projeto de compensação do licenciamento federal (IBAMA, 2010, 2012)"),
        "tecnico.fcr": Fonte("bibliografico", REF_KUBITZA),
        "tecnico.mortalidade_pct": Fonte("bibliografico", REF_KUBITZA),
        "tecnico.peso_final_g": Fonte("bibliografico", REF_KUBITZA),
        "tecnico.ciclo_dias": Fonte("bibliografico", REF_KUBITZA),
        "tecnico.produtividade_esperada_kg_m2_ciclo": Fonte("bibliografico", REF_KUBITZA),
        "tecnico.area_lamina_m2": Fonte("autor", "Dimensionamento conceitual (sem definição de terreno)"),
        "economico.preco_venda_kg": Fonte("cotacao", REF_COTACAO),
        "economico.custo_racao_kg": Fonte("cotacao", REF_COTACAO),
        "economico.custo_alevino_milheiro": Fonte("cotacao", REF_COTACAO),
        "economico.tarifa_energia_kwh": Fonte("cotacao", REF_COTACAO),
        "economico.classe_estimativa_aace": Fonte("normativo", "AACE International (2020), estimativa Classe 5"),
        "economico.tma_aa_pct": Fonte("autor", "Custo de oportunidade de recursos de fomento"),
        "ambiental.n_max_mg_l": Fonte("normativo", REF_CONAMA),
        "ambiental.p_max_mg_l": Fonte("normativo", REF_CONAMA),
        "ambiental.faixa_app_minima_m": Fonte("normativo", "Lei nº 12.651/2012, art. 4º, I, a"),
        "ambiental.proteina_racao_pct": Fonte("bibliografico", REF_KUBITZA),
        "ambiental.evaporacao_mm_dia": Fonte("bibliografico", "Kubitza (2017); calibrar com dados climáticos locais"),
        "social.salario_minimo": Fonte("normativo", "Salário mínimo nacional vigente"),
    }
    return cfg


def criar_alternativas() -> dict:
    """Três alternativas de projeto para comparação por TOPSIS."""
    a = criar_config_caso_base()
    a.projeto = "A — Viveiros semi-intensivos, energia da rede"

    b = criar_config_caso_base()
    b.projeto = "B — A + energia solar + wetland para efluentes"
    b.economico.capex_itens["Usina fotovoltaica"] = 55000.0
    b.economico.capex_itens["Wetland construído"] = 25000.0
    b.economico.tarifa_energia_kwh = 0.12           # custo residual (disponibilidade, O&M)
    b.monte_carlo.distribuicoes["economico.tarifa_energia_kwh"] = Distribuicao("triangular", 0.08, 0.12, 0.18)
    b.ambiental.fonte_energia = "solar"
    b.ambiental.remocao_p_tratamento_pct = 80.0
    b.ambiental.remocao_n_tratamento_pct = 60.0
    b.ambiental.checklist["energia_renovavel"] = True
    b.ambiental.checklist["gestao_residuos"] = True
    b.ambiental.residuos_reaproveitados_pct = 70.0

    c = criar_config_caso_base()
    c.projeto = "C — Intensificação com maior aeração"
    c.tecnico.sistema_produtivo = "intensivo"
    c.tecnico.produtividade_esperada_kg_m2_ciclo = 2.0
    c.economico.capex_itens["Aeradores e quadro elétrico"] = 140000.0
    c.economico.mao_obra_mes = 14000.0
    c.monte_carlo.distribuicoes["tecnico.mortalidade_pct"] = Distribuicao("triangular", 6.0, 12.0, 25.0)
    c.tecnico.mortalidade_pct = 12.0
    return {"A": a, "B": b, "C": c}


def criar_config_legado_rev186() -> EVTEASConfig:
    """Reproduz as premissas do preset da Rev. 186 (teste de regressão)."""
    cfg = EVTEASConfig(projeto="Preset legado da Rev. 186")
    cfg.tecnico = TecnicoConfig(sistema_produtivo="superintensivo", area_lamina_m2=1500, profundidade_media_m=1.0,
                                numero_tanques=10, tanques_ativos=9, capacidade_suporte_kg_m3=30.0,
                                produtividade_esperada_kg_m2_ciclo=8.0, peso_inicial_g=30, peso_final_g=850,
                                ciclo_dias=240, ciclos_ano=1.4, mortalidade_pct=8.0, fcr=1.60)
    cfg.economico = EconomicoConfig(
        tipo_organizacao="empresa", capex_itens={"CAPEX (preset)": 250000.0}, capital_giro=0.0,
        vida_util_anos=0.0, valor_residual_pct=0.0, preco_venda_kg=9.50, custo_racao_kg=3.20,
        custo_alevino_milheiro=450.0, tarifa_energia_kwh=0.0, outros_custos_variaveis_kg=0.0,
        mao_obra_mes=42000.0, encargos_mao_obra_pct=0.0, assistencia_tecnica_mes=25000.0,
        administrativo_mes=20000.0, manutencao_capex_pct_aa=0.0, taxa_impostos_faturamento_pct=15.0,
        taxa_impostos_lucro_pct=0.0, tma_aa_pct=12.0, horizonte_anos=10, capacidade_inicial_pct=50.0,
        meses_rampa=12, cooperados_trabalhadores=0)
    cfg.ambiental = AmbientalConfig(energia_kwh_kg=1.2)
    cfg.social = SocialConfig(empregos_diretos=10, empregos_indiretos=15, mao_obra_local_pct=80,
                              compras_locais_pct=60, conformidade_nr_pct=95, conflitos_registrados_ano=1)
    cfg.monte_carlo = MonteCarloConfig(distribuicoes={
        "economico.preco_venda_kg": Distribuicao("triangular", 8.0, 9.5, 11.0),
        "tecnico.fcr": Distribuicao("triangular", 1.35, 1.60, 1.90),
        "tecnico.mortalidade_pct": Distribuicao("triangular", 4.0, 8.0, 14.0),
        "economico.custo_racao_kg": Distribuicao("triangular", 2.70, 3.20, 4.00),
        "economico.custos_fixos_fator": Distribuicao("triangular", 0.90, 1.00, 1.15),
    }, incluir_capex_aace=False)
    cfg.monte_carlo.distribuicoes["economico.capex_fator"] = Distribuicao("triangular", 0.90, 1.00, 1.20)
    return cfg


def exemplo_validacao_especialistas(cfg: EVTEASConfig) -> EVTEASConfig:
    """Ilustra a parametrização por Likert (Quadro 7) com dois avaliadores.

    Os escores abaixo são *placeholders* de demonstração; devem ser
    substituídos pelas respostas efetivas dos especialistas na Fase 3.
    """
    c = copy.deepcopy(cfg)
    c.pesos = PesosConfig(metodo="likert", likert_dimensoes={
        "Especialista em framework": {"tecnico": 4, "economico": 5, "ambiental": 4, "social": 3},
        "Especialista em piscicultura": {"tecnico": 5, "economico": 5, "ambiental": 3, "social": 4},
    })
    return c

## 9. Resumo executivo, exportação e gráficos

In [ ]:
"""Resumo executivo, exportação (Excel/JSON/Markdown) e gráficos."""

import json
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd


def moeda(v) -> str:
    if v is None or not np.isfinite(v):
        return "n/d"
    return f"R$ {v:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def percentual(v, casas: int = 1) -> str:
    if v is None or not np.isfinite(v):
        return "n/d"
    return f"{v * 100:.{casas}f}%".replace(".", ",")


def numero(v, casas: int = 2) -> str:
    if v is None or not np.isfinite(v):
        return "n/d"
    return f"{v:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def resumo_executivo(r: Dict[str, Any]) -> pd.DataFrame:
    t, e, a, s, i = r["tecnico"], r["economico"], r["ambiental"], r["social"], r["indice"]
    linhas = [
        ("Classificação", r["decisao"]["classificacao"]),
        ("Produção anual (kg)", numero(t["producao_kg_ano"], 0)),
        ("OEE aquícola", percentual(t["oee"])),
        ("FCR (kg ração/kg ganho)", numero(t["fcr"])),
        ("Investimento total", moeda(e["investimento_total"])),
        ("VPL (projeto)", moeda(e["vpl"])),
        ("TIR a.a. (projeto)", percentual(e["tir_anual"]) + f" [{e['diagnostico_tir']}]"),
        ("Payback descontado (meses)", numero(e["payback_descontado_meses"], 1)),
        ("VPL (beneficiário, com fomento)", moeda(e["vpl_beneficiario"])),
        ("ROI anual em regime", percentual(e["roi_anual_regime"])),
        ("Margem de segurança", percentual(e["margem_seguranca"])),
        ("Custo total (R$/kg)", moeda(e["custo_total_kg"])),
        ("Pegada hídrica azul (m³/t)", numero(a["ph_azul_m3_t"], 0)),
        ("Pegada hídrica cinza (m³/t)", numero(a["ph_cinza_m3_t"], 0) + f" [{a['poluente_critico']}]"),
        ("Intensidade de carbono (kgCO2e/kg)", numero(a["intensidade_carbono_kgco2e_kg"])),
        ("Ecoeficiência (R$ VA/kgCO2e)", numero(a["ecoeficiencia_r_por_kgco2e"]) if a["ecoeficiencia_status"] == "definida"
         else a["ecoeficiencia_status"]),
        ("Conformidade ambiental (0–10)", numero(a["conformidade_0_10"], 1)),
        ("LSO (0–100)", numero(s["lso_0_100"], 1)),
        ("RVL (%)", numero(s["rvl_pct"], 1)),
        ("Renda mensal por trabalhador", moeda(s["renda_mensal_trabalhador"])),
        ("Índice EVTEAS (0–1)", numero(i["indice_evteas"], 3)),
        ("Score ESG (0–1)", numero(i["esg_score"], 3)),
    ]
    if "monte_carlo" in r:
        st = r["monte_carlo"]["estatisticas"]
        linhas += [("P(VPL > 0) — Monte Carlo", percentual(st["prob_vpl_positivo"])),
                   ("VPL médio — Monte Carlo", moeda(st["media_vpl"])),
                   ("P(VPL beneficiário > 0)", percentual(st["prob_vpl_beneficiario_positivo"]))]
    return pd.DataFrame(linhas, columns=["Indicador", "Valor"])


def serializar(obj):
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, pd.Series):
        return obj.to_dict()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    if isinstance(obj, float) and not np.isfinite(obj):
        return None
    if isinstance(obj, dict):
        return {str(k): serializar(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [serializar(v) for v in obj]
    return obj


def _df_dict(d: Dict[str, Any]) -> pd.DataFrame:
    return pd.DataFrame([{"Indicador": k, "Valor": v} for k, v in d.items()
                         if not isinstance(v, (dict, list, pd.DataFrame, np.ndarray))])


def exportar(r: Dict[str, Any], pasta="evteas_output") -> List[str]:
    pasta = Path(pasta)
    pasta.mkdir(parents=True, exist_ok=True)
    arquivos = []
    xlsx = pasta / "EVTEAS_Py_resultados.xlsx"
    with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
        resumo_executivo(r).to_excel(w, sheet_name="Resumo", index=False)
        r["premissas"].to_excel(w, sheet_name="Premissas", index=False)
        _df_dict(r["tecnico"]).to_excel(w, sheet_name="Tecnico", index=False)
        r["curva_crescimento"].to_excel(w, sheet_name="Curva_Crescimento", index=False)
        _df_dict({k: v for k, v in r["economico"].items() if k != "aliquotas"}).to_excel(w, sheet_name="Economico", index=False)
        r["dre_mensal"].to_excel(w, sheet_name="DRE_Mensal", index=False)
        r["dre_anual"].to_excel(w, sheet_name="DRE_Anual", index=False)
        pd.DataFrame({"Mês": range(len(r["fluxo_caixa"])), "Fluxo de caixa": r["fluxo_caixa"]}).to_excel(
            w, sheet_name="Fluxo_Caixa", index=False)
        _df_dict(r["ambiental"]).to_excel(w, sheet_name="Ambiental", index=False)
        pd.DataFrame(r["ambiental"]["conformidade_detalhe"]).to_excel(w, sheet_name="Conformidade", index=False)
        _df_dict(r["social"]).to_excel(w, sheet_name="Social", index=False)
        r["ods"].to_excel(w, sheet_name="ODS", index=False)
        pd.DataFrame([{"Desperdício": k, "R$/ano": v} for k, v in r["lean_green"]["desperdicios_r_ano"].items()]
                     ).to_excel(w, sheet_name="Lean_Green", index=False)
        pd.DataFrame({"KPI": list(r["indice"]["kpis_brutos"]),
                      "Valor bruto": list(r["indice"]["kpis_brutos"].values()),
                      "Normalizado": [r["indice"]["kpis_normalizados"][k] for k in r["indice"]["kpis_brutos"]]}
                     ).to_excel(w, sheet_name="KPIs", index=False)
        pd.DataFrame([{"Dimensão": k, "Score": v, "Peso": r["indice"]["pesos"]["dimensoes"][k]}
                      for k, v in r["indice"]["dimensoes"].items()]).to_excel(w, sheet_name="Indice", index=False)
        if "monte_carlo" in r:
            mc = r["monte_carlo"]
            mc["amostra"].to_excel(w, sheet_name="MC_Amostra", index=False)
            _df_dict({k: (str(v) if isinstance(v, tuple) else v) for k, v in mc["estatisticas"].items()}).to_excel(
                w, sheet_name="MC_Estatisticas", index=False)
            mc["convergencia"].to_excel(w, sheet_name="MC_Convergencia", index=False)
            mc["importancia"].to_excel(w, sheet_name="MC_Importancia", index=False)
        for chave, aba in (("sensibilidade", "Sensibilidade"), ("valores_criticos", "Valores_Criticos"),
                           ("cenarios", "Cenarios")):
            if chave in r:
                r[chave].to_excel(w, sheet_name=aba, index=False)
        pd.DataFrame(r["vv"]["verificacoes"]).to_excel(w, sheet_name="VV", index=False)
    arquivos.append(str(xlsx))
    js = pasta / "EVTEAS_Py_resultados.json"
    leve = {k: v for k, v in r.items() if k not in ("dre_mensal",)}
    if "monte_carlo" in leve:
        leve = dict(leve)
        leve["monte_carlo"] = {k: v for k, v in r["monte_carlo"].items() if k != "amostra"}
    with open(js, "w", encoding="utf-8") as f:
        json.dump(serializar(leve), f, ensure_ascii=False, indent=2, default=str)
    arquivos.append(str(js))
    md = pasta / "EVTEAS_Py_relatorio.md"
    md.write_text(relatorio_markdown(r), encoding="utf-8")
    arquivos.append(str(md))
    return arquivos


def relatorio_markdown(r: Dict[str, Any]) -> str:
    linhas = [f"# Relatório EVTEAS-Py — {r['config']['projeto']}", "", f"Versão: {r['versao']}", "",
              "## Resumo executivo", "", "| Indicador | Valor |", "|---|---|"]
    linhas += [f"| {a} | {b} |" for a, b in resumo_executivo(r).itertuples(index=False)]
    d = r["decisao"]
    linhas += ["", "## Decisão", "", f"**{d['classificacao']}**", ""]
    linhas += [f"- Veto: {v}" for v in d["vetos"]] + [f"- Ressalva: {v}" for v in d["ressalvas"]]
    linhas += ["", "## ODS atendidos", ""] + [f"- {o}" for o, ok in zip(r["ods"]["ODS"], r["ods"]["Atendido"]) if ok]
    vv = r["vv"]
    linhas += ["", "## Verificação", "", f"{vv['quantidade_verificacoes']} verificações; {vv['quantidade_erros']} falha(s)."]
    return "\n".join(linhas) + "\n"


# ---------------------------------------------------------------------------
# Gráficos
# ---------------------------------------------------------------------------

def _plt():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.spines.top": False,
                         "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3,
                         "figure.dpi": 100, "savefig.dpi": 220})
    return plt


AZUL, LARANJA, VERDE, CINZA, VERMELHO = "#2a6f97", "#e07a1f", "#4c956c", "#6c757d", "#c0392b"


def gerar_graficos(r: Dict[str, Any], pasta="evteas_output/figuras", comparacao: Dict[str, Any] = None) -> Dict[str, str]:
    plt = _plt()
    pasta = Path(pasta)
    pasta.mkdir(parents=True, exist_ok=True)
    figs: Dict[str, str] = {}

    def salvar(fig, nome):
        p = pasta / nome
        fig.tight_layout()
        fig.savefig(p, bbox_inches="tight")
        plt.close(fig)
        figs[nome.split(".")[0]] = str(p)

    # 1. Curva de crescimento
    cc = r["curva_crescimento"]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(cc["Dia"], cc["Peso médio (g)"], color=AZUL, lw=2, label="Peso médio (g)")
    ax.set_xlabel("Dia do ciclo"); ax.set_ylabel("Peso médio (g)", color=AZUL)
    ax2 = ax.twinx(); ax2.grid(False)
    ax2.plot(cc["Dia"], cc["Biomassa (kg)"] / 1000, color=LARANJA, lw=2, label="Biomassa (t)")
    ax2.plot(cc["Dia"], cc["Ração acumulada (kg)"] / 1000, color=CINZA, lw=1.5, ls="--", label="Ração acumulada (t)")
    ax2.set_ylabel("t", color=LARANJA)
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="upper left", frameon=False)
    ax.set_title("Curva de crescimento, biomassa e arraçoamento por ciclo")
    salvar(fig, "01_curva_crescimento.png")

    # 2. Receita, custos e fluxo acumulado
    df = r["dre_mensal"]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df["Mês"], df["Receita Bruta"] / 1000, color=AZUL, lw=1.8, label="Receita bruta")
    ax.plot(df["Mês"], (df["Custos Variáveis"] + df["Custos Fixos"]) / 1000, color=LARANJA, lw=1.8, label="Custos operacionais")
    ax.set_xlabel("Mês"); ax.set_ylabel("R$ mil/mês")
    ax2 = ax.twinx(); ax2.grid(False)
    ax2.plot(range(len(r["fluxo_caixa"])), np.cumsum(r["fluxo_caixa"]) / 1000, color=VERDE, lw=2, label="Fluxo de caixa acumulado")
    ax2.axhline(0, color=CINZA, lw=0.8)
    ax2.set_ylabel("R$ mil (acumulado)", color=VERDE)
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="lower right", frameon=False)
    ax.set_title("Projeção mensal: receita, custos e fluxo de caixa acumulado")
    salvar(fig, "02_projecao_economica.png")

    # 3. Scores dimensionais
    dims = r["indice"]["dimensoes"]
    rot = {"tecnico": "Técnica", "economico": "Econômica", "ambiental": "Ambiental", "social": "Social"}
    fig, ax = plt.subplots(figsize=(7, 3.6))
    vals = [dims[k] for k in rot]
    ax.barh(list(rot.values()), vals, color=[AZUL, LARANJA, VERDE, CINZA])
    for y, v in enumerate(vals):
        ax.text(v + 0.01, y, f"{v:.3f}".replace(".", ","), va="center")
    ax.axvline(r["indice"]["indice_evteas"], color=VERMELHO, ls="--", lw=1.2,
               label=f"Índice EVTEAS = {r['indice']['indice_evteas']:.3f}".replace(".", ","))
    ax.set_xlim(0, 1); ax.set_xlabel("Score normalizado (0–1)"); ax.invert_yaxis()
    ax.legend(frameon=False, loc="upper right", bbox_to_anchor=(1.0, -0.18)); ax.set_title("Scores dimensionais e índice integrado")
    salvar(fig, "03_dimensoes.png")

    # 4. Radar social
    radar = r["social"]["radar_social"]
    nomes = list(radar); valores = [radar[k] for k in nomes]
    ang = np.linspace(0, 2 * np.pi, len(nomes), endpoint=False).tolist()
    fig = plt.figure(figsize=(5.2, 5.2)); ax = fig.add_subplot(111, polar=True)
    ax.plot(ang + ang[:1], valores + valores[:1], color=VERDE, lw=2)
    ax.fill(ang + ang[:1], valores + valores[:1], color=VERDE, alpha=0.25)
    ax.set_xticks(ang); ax.set_xticklabels(nomes, fontsize=9); ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0]); ax.set_yticklabels(["0,25", "0,50", "0,75", "1,00"], fontsize=7, color=CINZA)
    ax.tick_params(axis="x", pad=8)
    ax.set_title("Radar Social", pad=18)
    salvar(fig, "04_radar_social.png")

    if "monte_carlo" in r:
        mc = r["monte_carlo"]; v = mc["amostra"]["VPL"] / 1000; st = mc["estatisticas"]
        fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
        axs[0].hist(v, bins=60, color=AZUL, alpha=0.85)
        axs[0].axvline(0, color=VERMELHO, ls="--", lw=1.2, label="VPL = 0")
        axs[0].axvline(r["economico"]["vpl"] / 1000, color=LARANJA, lw=1.5, label="Determinístico (moda)")
        axs[0].axvline(st["media_vpl"] / 1000, color=VERDE, lw=1.5, label="Média Monte Carlo")
        axs[0].set_xlabel("VPL (R$ mil)"); axs[0].set_ylabel("Frequência"); axs[0].legend(frameon=False, fontsize=8)
        axs[0].set_title("Distribuição do VPL")
        xs = np.sort(v); axs[1].plot(xs, np.arange(1, len(xs) + 1) / len(xs), color=AZUL, lw=2)
        axs[1].axvline(0, color=VERMELHO, ls="--", lw=1.2)
        axs[1].set_xlabel("VPL (R$ mil)"); axs[1].set_ylabel("Probabilidade acumulada")
        axs[1].set_title(f"P(VPL ≤ 0) = {st['prob_vpl_negativo']:.1%}".replace(".", ","))
        salvar(fig, "05_monte_carlo_vpl.png")

        conv = mc["convergencia"]
        fig, ax = plt.subplots(figsize=(7, 3.6))
        ax.errorbar(conv["Iterações"], conv["Média VPL"] / 1000, yerr=1.96 * conv["Erro-padrão"] / 1000,
                    color=AZUL, marker="o", ms=3, capsize=3)
        ax.set_xscale("log"); ax.set_xlabel("Iterações (escala log)"); ax.set_ylabel("Média do VPL (R$ mil) ± IC95%")
        ax.set_title("Convergência da simulação de Monte Carlo")
        salvar(fig, "06_convergencia_mc.png")

    if "sensibilidade" in r:
        s = r["sensibilidade"]; s = s[s["Amplitude"] == s["Amplitude"].min()].copy()
        s = s.sort_values("Amplitude do efeito")
        rotulo = s["Variável"].str.split(".").str[1]
        fig, ax = plt.subplots(figsize=(8, 4.2))
        ax.barh(rotulo, s["Δ VPL (−)"] / 1000, color=LARANJA, label="−10%")
        ax.barh(rotulo, s["Δ VPL (+)"] / 1000, color=AZUL, label="+10%")
        ax.axvline(0, color="black", lw=0.8)
        ax.set_xlabel("Variação do VPL (R$ mil)"); ax.legend(frameon=False)
        ax.set_title("Diagrama de tornado — sensibilidade do VPL a ±10%")
        salvar(fig, "07_tornado.png")

    if comparacao is not None:
        rk = comparacao["ranking"]
        fig, ax = plt.subplots(figsize=(7, 3.2))
        ax.barh(rk["Alternativa"], rk["TOPSIS"], color=[VERDE, AZUL, CINZA][: len(rk)])
        for y, v in enumerate(rk["TOPSIS"]):
            ax.text(v + 0.01, y, f"{v:.3f}".replace(".", ","), va="center")
        ax.set_xlim(0, 1); ax.invert_yaxis(); ax.set_xlabel("Proximidade relativa à solução ideal (TOPSIS)")
        ax.set_title("Comparação de alternativas de projeto")
        salvar(fig, "08_topsis.png")
    return figs


def diagrama_arquitetura(caminho="evteas_output/figuras/00_arquitetura.png") -> str:
    """Diagrama do fluxo Input → Processamento → Output → Verificação."""
    plt = _plt()
    from matplotlib.patches import FancyBboxPatch
    fig, ax = plt.subplots(figsize=(10, 4.6)); ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 5)
    caixas = [
        (0.1, 3.3, 2.0, 1.4, "Entradas\nEVTEASConfig\n+ registro de fontes", "#dbe9f6"),
        (2.6, 3.3, 2.2, 1.4, "Motor vetorizado\ntécnico → econômico\n→ ambiental → social", "#fde8d0"),
        (5.3, 3.3, 2.0, 1.4, "Normalização\npesos (preset/\nLikert/AHP)", "#e3f1e6"),
        (7.8, 3.3, 2.1, 1.4, "Índice EVTEAS\nESG, ODS\nregra de decisão", "#eee"),
        (2.6, 0.6, 2.2, 1.6, "Incerteza\nMonte Carlo (n = 10.000)\nsensibilidade, cenários,\nvalores críticos", "#fde8d0"),
        (5.3, 0.6, 2.0, 1.6, "Verificação (V&V)\ninvariantes e\ntestes automatizados", "#e3f1e6"),
        (7.8, 0.6, 2.1, 1.6, "Saídas\nExcel, JSON,\nrelatório e figuras\nTOPSIS", "#eee"),
    ]
    for x, y, w, h, txt, cor in caixas:
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05", fc=cor, ec="#555", lw=1))
        ax.text(x + w / 2, y + h / 2, txt, ha="center", va="center", fontsize=9)
    seta = dict(arrowstyle="->", color="#333", lw=1.2)
    for (x1, y1), (x2, y2) in [((2.15, 4.0), (2.55, 4.0)), ((4.85, 4.0), (5.25, 4.0)), ((7.35, 4.0), (7.75, 4.0)),
                               ((3.7, 3.25), (3.7, 2.25)), ((4.85, 1.4), (5.25, 1.4)), ((7.35, 1.4), (7.75, 1.4)),
                               ((8.85, 3.25), (8.85, 2.25))]:
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=seta)
    ax.set_title("Arquitetura do EVTEAS-Py 2.0", fontsize=12)
    Path(caminho).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(caminho, bbox_inches="tight"); plt.close(fig)
    return caminho

## 10. Entrada interativa (wizard), arquivos JSON e análise de preços

In [ ]:
"""Interfaces de entrada: wizard interativo, arquivos JSON e análise de preços.

O wizard preserva o percurso de coleta das versões anteriores (Viabilidade
econômica Rev. 8/9 e EVTEAS master): tipo de organização, mix de produtos,
CAPEX, custos, cooperados, tributos, rampa e parâmetros ESG. Toda pergunta
aceita Enter para manter o valor exibido entre colchetes.
"""

import json
from pathlib import Path
from typing import Any, Callable, Dict, Optional

import numpy as np
import pandas as pd



def salvar_config(cfg: EVTEASConfig, caminho) -> str:
    Path(caminho).write_text(json.dumps(config_para_dict(cfg), ensure_ascii=False, indent=2), encoding="utf-8")
    return str(caminho)


def carregar_config(caminho) -> EVTEASConfig:
    return config_de_dict(json.loads(Path(caminho).read_text(encoding="utf-8")))


def _num(txt: str) -> float:
    txt = txt.strip().replace("R$", "").replace("%", "").strip()
    if "," in txt and "." in txt:
        txt = txt.replace(".", "").replace(",", ".") if txt.rfind(",") > txt.rfind(".") else txt.replace(",", "")
    elif "," in txt:
        txt = txt.replace(",", ".")
    return float(txt)


def perguntar_numero(rotulo: str, atual: float, minimo: Optional[float] = 0.0, maximo: Optional[float] = None,
                     entrada: Callable[[str], str] = input) -> float:
    while True:
        txt = entrada(f"{rotulo} [{atual}]: ").strip()
        if not txt:
            return atual
        try:
            v = _num(txt)
        except ValueError:
            print("  Valor inválido; use números (vírgula ou ponto decimal).")
            continue
        if minimo is not None and v < minimo:
            print(f"  Valor deve ser ≥ {minimo}.")
        elif maximo is not None and v > maximo:
            print(f"  Valor deve ser ≤ {maximo}.")
        else:
            return v


def perguntar_opcao(rotulo: str, atual: str, opcoes, entrada: Callable[[str], str] = input) -> str:
    while True:
        txt = entrada(f"{rotulo} {list(opcoes)} [{atual}]: ").strip()
        if not txt:
            return atual
        if txt in opcoes:
            return txt
        print("  Opção inválida.")


def perguntar_sim_nao(rotulo: str, atual: bool, entrada: Callable[[str], str] = input) -> bool:
    while True:
        txt = entrada(f"{rotulo} (s/n) [{'s' if atual else 'n'}]: ").strip().lower()
        if not txt:
            return atual
        if txt in ("s", "n"):
            return txt == "s"


def wizard_evteas(entrada: Callable[[str], str] = input, base: Optional[EVTEASConfig] = None) -> EVTEASConfig:
    """Coleta guiada das premissas, bloco a bloco (lógica PMBOK de iniciação/planejamento)."""
    cfg = base or criar_config_caso_base()
    t, e, a, s = cfg.tecnico, cfg.economico, cfg.ambiental, cfg.social
    print("=== EVTEAS-Py — entrada de dados (Enter mantém o valor) ===")
    cfg.projeto = entrada(f"Nome do projeto [{cfg.projeto}]: ").strip() or cfg.projeto

    print("\n--- 1. Técnico: infraestrutura ---")
    t.sistema_produtivo = perguntar_opcao("Sistema produtivo", t.sistema_produtivo,
                                          ["extensivo", "semi-intensivo", "intensivo", "superintensivo"], entrada)
    t.area_lamina_m2 = perguntar_numero("Área de lâmina d'água (m²)", t.area_lamina_m2, 1, None, entrada)
    t.profundidade_media_m = perguntar_numero("Profundidade média (m)", t.profundidade_media_m, 0.1, 10, entrada)
    t.numero_tanques = int(perguntar_numero("Número de viveiros/tanques", t.numero_tanques, 1, None, entrada))
    t.tanques_ativos = int(perguntar_numero("Viveiros ativos", t.tanques_ativos, 0, t.numero_tanques, entrada))
    print("--- 1. Técnico: parâmetros biológicos e insumos ---")
    t.peso_inicial_g = perguntar_numero("Peso inicial dos alevinos/juvenis (g)", t.peso_inicial_g, 0.1, None, entrada)
    t.peso_final_g = perguntar_numero("Peso de abate (g)", t.peso_final_g, t.peso_inicial_g, None, entrada)
    t.ciclo_dias = int(perguntar_numero("Duração do ciclo (dias)", t.ciclo_dias, 1, 730, entrada))
    t.ciclos_ano = perguntar_numero("Ciclos por ano (escalonados)", t.ciclos_ano, 0, 365 / t.ciclo_dias, entrada)
    t.produtividade_esperada_kg_m2_ciclo = perguntar_numero("Produtividade esperada (kg/m²/ciclo; 0 ignora)",
                                                            t.produtividade_esperada_kg_m2_ciclo, 0, None, entrada)
    t.fcr = perguntar_numero("Conversão alimentar — FCR/TCA", t.fcr, 0.5, 5, entrada)
    t.mortalidade_pct = perguntar_numero("Mortalidade (%)", t.mortalidade_pct, 0, 100, entrada)

    print("\n--- 2. Econômico ---")
    e.tipo_organizacao = perguntar_opcao("Tipo de organização", e.tipo_organizacao,
                                         ["empresa", "coop_trabalho", "coop_agro"], entrada)
    if perguntar_sim_nao("Detalhar itens de CAPEX?", False, entrada):
        itens = {}
        while True:
            nome = entrada("  Item de CAPEX (Enter encerra): ").strip()
            if not nome:
                break
            itens[nome] = perguntar_numero(f"  Valor de '{nome}' (R$)", 0.0, 0, None, entrada)
        if itens:
            e.capex_itens = itens
    e.classe_estimativa_aace = int(perguntar_numero("Classe da estimativa AACE (1 a 5)", e.classe_estimativa_aace, 1, 5, entrada))
    e.capital_giro = perguntar_numero("Capital de giro (R$)", e.capital_giro, 0, None, entrada)
    e.fomento_nao_reembolsavel_pct = perguntar_numero("Fomento não reembolsável (% do CAPEX)",
                                                      e.fomento_nao_reembolsavel_pct, 0, 100, entrada)
    if perguntar_sim_nao("Informar mix de produtos?", bool(e.mix_produtos), entrada):
        mix = []
        k = int(perguntar_numero("  Quantidade de produtos", max(len(e.mix_produtos), 1), 1, None, entrada))
        for j in range(k):
            nome = entrada(f"  Nome do produto {j + 1}: ").strip() or f"Produto {j + 1}"
            mix.append({"nome": nome,
                        "participacao_pct": perguntar_numero(f"  Participação de '{nome}' no volume (%)", 100 / k, 0, 100, entrada),
                        "preco_kg": perguntar_numero(f"  Preço de '{nome}' (R$/kg)", e.preco_venda_kg, 0, None, entrada)})
        e.mix_produtos = mix
    else:
        e.preco_venda_kg = perguntar_numero("Preço de venda (R$/kg)", e.preco_venda_kg, 0, None, entrada)
    e.custo_racao_kg = perguntar_numero("Custo da ração (R$/kg)", e.custo_racao_kg, 0, None, entrada)
    e.custo_alevino_milheiro = perguntar_numero("Custo do milheiro de alevinos (R$)", e.custo_alevino_milheiro, 0, None, entrada)
    e.tarifa_energia_kwh = perguntar_numero("Tarifa de energia (R$/kWh)", e.tarifa_energia_kwh, 0, None, entrada)
    e.mao_obra_mes = perguntar_numero("Mão de obra / retiradas (R$/mês)", e.mao_obra_mes, 0, None, entrada)
    e.encargos_mao_obra_pct = perguntar_numero("Encargos sobre a mão de obra (%)", e.encargos_mao_obra_pct, 0, 100, entrada)
    if e.tipo_organizacao != "empresa":
        minimo = 7 if e.tipo_organizacao == "coop_trabalho" else 20
        print(f"  (Lei nº 12.690/2012 e Lei nº 5.764/1971: mínimo usual de {minimo} cooperados no total)")
        e.cooperados_trabalhadores = int(perguntar_numero("Cooperados que trabalham", e.cooperados_trabalhadores, 0, None, entrada))
        e.cooperados_fornecedores = int(perguntar_numero("Cooperados fornecedores", e.cooperados_fornecedores, 0, None, entrada))
        e.perc_sobras_trabalhadores = perguntar_numero("% das sobras aos trabalhadores", e.perc_sobras_trabalhadores, 0, 100, entrada)
    e.taxa_impostos_faturamento_pct = perguntar_numero("Tributos sobre faturamento (%)", e.taxa_impostos_faturamento_pct, 0, 100, entrada)
    e.taxa_impostos_lucro_pct = perguntar_numero("Tributos sobre lucro (%)", e.taxa_impostos_lucro_pct, 0, 100, entrada)
    e.tma_aa_pct = perguntar_numero("TMA (% a.a.)", e.tma_aa_pct, 0, 100, entrada)
    e.horizonte_anos = int(perguntar_numero("Horizonte (anos)", e.horizonte_anos, 1, 50, entrada))

    print("\n--- 3. Ambiental ---")
    a.fonte_energia = perguntar_opcao("Fonte de energia", a.fonte_energia, ["rede", "solar", "biomassa", "mista"], entrada)
    a.remocao_p_tratamento_pct = perguntar_numero("Remoção de fósforo no tratamento (%)", a.remocao_p_tratamento_pct, 0, 100, entrada)
    a.distancia_app_m = perguntar_numero("Distância da APP (m)", a.distancia_app_m, 0, None, entrada)
    for k in list(a.checklist):
        a.checklist[k] = perguntar_sim_nao(f"Atende: {k.replace('_', ' ')}?", a.checklist[k], entrada)

    print("\n--- 4. Social ---")
    s.empregos_diretos = int(perguntar_numero("Empregos diretos", s.empregos_diretos, 0, None, entrada))
    s.empregos_indiretos = int(perguntar_numero("Empregos indiretos", s.empregos_indiretos, 0, None, entrada))
    s.mao_obra_local_pct = perguntar_numero("Mão de obra local (%)", s.mao_obra_local_pct, 0, 100, entrada)
    s.compras_locais_pct = perguntar_numero("Compras locais (%)", s.compras_locais_pct, 0, 100, entrada)
    s.conformidade_nr_pct = perguntar_numero("Conformidade com NR (%)", s.conformidade_nr_pct, 0, 100, entrada)
    s.relacao_comunidade = perguntar_opcao("Relação com a comunidade", s.relacao_comunidade,
                                           ["inexistente", "parcial", "consolidado"], entrada)
    s.reunioes_comunidade_ano = int(perguntar_numero("Reuniões com a comunidade por ano", s.reunioes_comunidade_ano, 0, None, entrada))
    s.conflitos_registrados_ano = int(perguntar_numero("Conflitos registrados por ano", s.conflitos_registrados_ano, 0, None, entrada))
    s.mulheres_pct = perguntar_numero("Participação feminina (%)", s.mulheres_pct, 0, 100, entrada)

    print("\n--- 5. Governança ---")
    for k in list(cfg.governanca.itens):
        cfg.governanca.itens[k] = perguntar_sim_nao(f"Atende: {k.replace('_', ' ')}?", cfg.governanca.itens[k], entrada)
    print("\nEntradas registradas.")
    return cfg


# ---------------------------------------------------------------------------
# Análise de preços a partir de planilha (herdada da versão anterior)
# ---------------------------------------------------------------------------

def analisar_precos(df: pd.DataFrame, col_preco: str, col_qtd: Optional[str] = None,
                    custo_unitario: float = 0.0, z_corte: float = 3.0) -> Dict[str, Any]:
    """Histórico de preços (estatísticas e estratégias) ou, com quantidades,
    curva de demanda linear e preço que maximiza a margem de contribuição."""
    d = df[[c for c in (col_preco, col_qtd) if c]].apply(pd.to_numeric, errors="coerce").dropna()
    d = d[d[col_preco] > 0]
    if len(d) > 10:
        z = ((d - d.mean()) / d.std(ddof=0)).abs()
        d = d[(z <= z_corte).all(axis=1)]
    if d.empty:
        raise ValueError("Sem dados válidos de preço")
    p = d[col_preco]
    out: Dict[str, Any] = {"n": int(len(d)), "estatisticas": p.describe().to_dict(),
                           "sugestoes": {"Econômico (P25)": float(p.quantile(.25)), "Mediana": float(p.median()),
                                         "Premium (P75)": float(p.quantile(.75))}}
    if col_qtd and len(d) >= 5:
        b, a = np.polyfit(p, d[col_qtd], 1)
        r2 = float(np.corrcoef(p, d[col_qtd])[0, 1] ** 2)
        out["demanda"] = {"intercepto": float(a), "inclinacao": float(b), "r2": r2}
        if b < 0:
            otimo = (custo_unitario - a / b) / 2   # max (P − c)(a + bP)
            out["preco_otimo"] = float(otimo)
    return out

## 11. Execução do caso-base

Para usar outro projeto: `cfg = wizard_evteas()` (entrada interativa) ou `cfg = carregar_config('meu_projeto.json')`.

In [ ]:
assert teste_regressao_deterministica()
cfg = criar_config_caso_base()
# cfg = wizard_evteas()            # entrada interativa completa
resultado = executar_evteas(cfg, executar_mc=True, executar_sens=True)
resumo_executivo(resultado)

In [ ]:
print(resultado['decisao'])
resultado['premissas'].head(20)

In [ ]:
resultado['dre_anual'].round(0)

In [ ]:
st = resultado['monte_carlo']['estatisticas']
{k: v for k, v in st.items() if not isinstance(v, dict)}

In [ ]:
resultado['monte_carlo']['importancia']

In [ ]:
display(resultado['valores_criticos'])
display(resultado['cenarios'])

In [ ]:
resultado['ods']

In [ ]:
import pandas as pd
pd.DataFrame(resultado['vv']['verificacoes'])

## 12. Comparação de alternativas (TOPSIS) e parametrização por especialistas (Likert)

In [ ]:
comparacao = comparar_alternativas(criar_alternativas(), n_mc=5000)
comparacao['ranking']

In [ ]:
r_likert = executar_evteas(exemplo_validacao_especialistas(cfg), executar_mc=False, executar_sens=False)
display(r_likert['indice']['pesos']['likert_dimensoes']['tabela'])
print(r_likert['decisao'])

## 13. Exportação e gráficos

In [ ]:
arquivos = exportar(resultado, 'evteas_output')
figuras = gerar_graficos(resultado, 'evteas_output/figuras', comparacao)
diagrama_arquitetura('evteas_output/figuras/00_arquitetura.png')
salvar_config(cfg, 'evteas_output/config_caso_base.json')
from IPython.display import Image
for f in sorted(figuras.values()):
    display(Image(f))
arquivos